# Phase 5 · Legal Statute Identification (LSI)

**Task:** multi-label classification. For each case, predict the set of relevant statute IDs. **Metric: macro-F1** over statute IDs (secondary: micro-F1, subset accuracy), so rare statutes matter as much as frequent ones. Output: `[{"id": …, "pred": [statute ids]}, …]`, with unique, valid IDs.

**Requirements (§11):** start from the Phase 1 models and use all four. At least 6 substantially different experiments: at least 5 fine-tuning techniques from {full, partial, QLoRA, **LoRMA (compulsory)**, AdaLoRA, adapters, prefix, prompt} plus at least 1 **KD**. Analyses: long-document representation, multi-label losses, class imbalance and rare statutes, **threshold strategies (global vs per-label)**, calibration, label co-occurrence, and model size versus inference cost. No external legal corpora or pretrained models.

**P5-0:** download; profile sizes, document lengths, the label space, labels per case and label frequencies; check for overlap between splits (Phase 4's test set was contained in dev); build clean splits and a one-time token cache.

In [1]:
# ============================================================================================
# P5-0 — Phase 5 download + profile + leakage check + clean splits + token cache (CPU, ~5-10 min)
#   Writes data_p5/clean/{train,dev,test}.json ({id, labels}) and data_p5/tok/{split}.npz (all tokens)
# ============================================================================================
import os, sys, json, glob, hashlib, random, time
import numpy as np
from collections import Counter, defaultdict
from multiprocessing import Pool
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

WORK, TOKEN = "/kaggle/working", UserSecretsClient().get_secret("HF_WRITE_TOKEN")
sys.path.insert(0, f"{WORK}/src")

# ---- 1. Phase-1 assets present? (restore with R1 if not) ----------------------------------------
need = [f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json", f"{WORK}/src/p4_cjpe.py"] + \
       [f"{WORK}/checkpoints/{m}/best.pt" for m in ["m1_dense_100m", "m2_mla_moe", "m3_encdec", "m4_copy"]]
print("assets OK:", all(map(os.path.exists, need)), "| missing:", [os.path.relpath(p, WORK) for p in need if not os.path.exists(p)])

# ---- 2. download + load ----------------------------------------------------------------------------
snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=TOKEN,
                  allow_patterns=["Phase_5_LSI/**"], local_dir=f"{WORK}/data_p5")
files = sorted(glob.glob(f"{WORK}/data_p5/**/*.json*", recursive=True))
for f in files:
    print(f"   {os.path.relpath(f, WORK)}  {os.path.getsize(f)/1e6:.1f} MB")
def load(path):
    if path.endswith(".jsonl"):
        return [json.loads(l) for l in open(path)]
    obj = json.load(open(path))
    return [{"id": k, **v} if isinstance(v, dict) else {"id": k, "text": v} for k, v in obj.items()] if isinstance(obj, dict) else obj
S = {os.path.basename(f).split(".")[0]: load(f) for f in files}
for n, recs in S.items():
    print(f"\n=== {n}: {len(recs)} records | keys {list(recs[0].keys())}")
    print("    example:", {k: (v[:150] + "…" if isinstance(v, str) and len(v) > 150 else v) for k, v in recs[0].items()})

# ---- 3. label space ------------------------------------------------------------------------------
lab = lambda r: sorted(set(r.get("labels") or []))
tr_c = Counter(l for r in S["train"] for l in lab(r))
dv_c = Counter(l for r in S["dev"] for l in lab(r))
print(f"\nstatute IDs: train {len(tr_c)} | dev {len(dv_c)} | in dev but not in train: {sorted(set(dv_c) - set(tr_c))}")
print(f"ID range: {min(tr_c)}..{max(tr_c)}")
for n in ("train", "dev"):
    k = np.array([len(lab(r)) for r in S[n]])
    print(f"labels per case ({n}): mean {k.mean():.2f} | median {int(np.median(k))} | max {k.max()} | "
          f"0 labels: {(k == 0).sum()} | 1: {(k == 1).sum()} | 2-3: {((k >= 2) & (k <= 3)).sum()} | 4+: {(k >= 4).sum()}")
freq = np.array(sorted(tr_c.values(), reverse=True))
print(f"train label frequency: top-5 {freq[:5].tolist()} | median {int(np.median(freq))} | "
      f"labels with <10 cases: {(freq < 10).sum()} | <50: {(freq < 50).sum()} | <100: {(freq < 100).sum()}")
print("most frequent statutes:", tr_c.most_common(10))
pairs = Counter(tuple(p) for r in S["train"] for i, a in enumerate(lab(r)) for p in [(a, b) for b in lab(r)[i + 1:]])
print("most frequent co-occurring pairs:", pairs.most_common(5))

# ---- 4. leakage / duplicates between splits ---------------------------------------------------
h = lambda t: hashlib.md5(t.encode()).hexdigest()
H = {n: {h(r["text"]) for r in recs} for n, recs in S.items()}56
I = {n: {str(r["id"]) for r in recs} for n, recs in S.items()}
names = list(S)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        print(f"overlap {a} vs {b}: ids {len(I[a] & I[b])} | identical texts {len(H[a] & H[b])}")
print("test has a non-empty labels field:", any(r.get("labels") for r in S["test"]))
print("duplicate texts inside train:", len(S["train"]) - len(H["train"]))

# ---- 5. clean splits: dev without test texts; train dedup (drop groups whose label sets disagree) --
test_h, test_ids = H["test"], I["test"]
dev = [r for r in S["dev"] if str(r["id"]) not in test_ids and h(r["text"]) not in test_h]
groups = defaultdict(list)
for r in S["train"]:
    groups[h(r["text"])].append(r)
banned = test_h | H["dev"]
train, conflicts = [], 0
for k, rs in groups.items():
    if k in banned:
        continue
    if len({tuple(lab(r)) for r in rs}) > 1:
        conflicts += 1; continue
    train.append(rs[0])
test = S["test"]
print(f"\nclean: train {len(train)} (dropped {len(S['train']) - len(train)}, label conflicts {conflicts}) | "
      f"dev {len(dev)} (was {len(S['dev'])}) | test {len(test)}")

# ---- 6. tokenise every case once (normalise + A1-BPE, 4 processes) ---------------------------
def _init():
    global _tk, _norm
    from tokenizers import Tokenizer
    from common import normalise as _n
    _tk = Tokenizer.from_file(f"{WORK}/tokenizer/hf_a1_bpe_v2_32k.json"); _norm = _n
def _enc(t):
    return np.asarray(_tk.encode(_norm(t), add_special_tokens=False).ids, dtype=np.uint16)
os.makedirs(f"{WORK}/data_p5/tok", exist_ok=True); os.makedirs(f"{WORK}/data_p5/clean", exist_ok=True)
for name, recs in (("dev", dev), ("test", test), ("train", train)):
    t0 = time.time()
    with Pool(4, initializer=_init) as pool:
        arrs = pool.map(_enc, [r["text"] for r in recs], chunksize=64)
    lens = np.array([len(a) for a in arrs], dtype=np.int64)
    np.savez(f"{WORK}/data_p5/tok/{name}.npz", ids=np.concatenate(arrs), offsets=np.concatenate([[0], np.cumsum(lens)]))
    json.dump([{"id": str(r["id"]), "labels": (lab(r) if name != "test" else None)} for r in recs],
              open(f"{WORK}/data_p5/clean/{name}.json", "w"))           # test labels are never stored
    print(f"   {name}: {len(recs)} cases, {lens.sum()/1e6:.1f}M tokens | median {int(np.median(lens))} "
          f"p90 {int(np.percentile(lens, 90))} max {lens.max()} | <=512: {np.mean(lens <= 512):.0%} | {time.time()-t0:.0f}s")
import shutil; print(f"free disk: {shutil.disk_usage(WORK).free/1e9:.1f} GB")

assets OK: True | missing: []


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

   data_p5/Phase_5_LSI/Test/test.json  98.1 MB
   data_p5/Phase_5_LSI/Train/dev.json  79.4 MB
   data_p5/Phase_5_LSI/Train/train.json  299.4 MB

=== test: 13019 records | keys ['id', 'text']
    example: {'id': '100002997', 'text': '2.Prosecution case in brief was that <ENTITY>, daughter of informant <ENTITY> (PW-1) was married to <ENTITY> 5 years ago. After marriage <ENTITY> was …'}

=== dev: 10181 records | keys ['id', 'text', 'labels']
    example: {'id': '1000139', 'text': 'Dr. <ENTITY>, J. The accused persons are the <ENTITY> andbrother respectively of the deceased. The incident purportedly took place on 3.1.1990.Though …', 'labels': [5]}

=== train: 42750 records | keys ['id', 'text', 'labels']
    example: {'id': '1000008', 'text': '(a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stated thus: With the growth of in…', 'labels': [69, 9, 3]}

statute IDs: train 100 | dev 100 | in dev but not in train: []
ID range: 

## P5-1 · Multi-label module (100 statutes)

Reuses the Phase-4 code (token cache, section views, document classifier, chunk encoder) with a 100-way sigmoid head. The loss is BCE with per-statute pos_weight = √(neg/pos); there is optional KD on tempered sigmoids and an “at least one label” fallback. Thresholds are one global value or one per statute; the per-statute version is cross-fitted on two halves of dev for an honest estimate. Everything is plain PyTorch with no PEFT library.

In [2]:
%%writefile /kaggle/working/src/p5_lsi.py
"""
p5_lsi.py -- Phase 5 (Legal Statute Identification, multi-label over 100 statute IDs).

Reuses the Phase-4 machinery (token cache, section views, DocClassifier, chunk encoder) with a
multi-label head:
  * 100 sigmoid outputs, BCE with per-label pos_weight = sqrt(#neg / #pos)  (rare statutes count more)
  * optional KD: BCE between the student's and the teacher's tempered sigmoids (x T^2)
  * decision rules: one global threshold, or one threshold per statute (tuned on dev), always at
    least one label per case (argmax fallback)
  * metrics: macro-F1 over the 100 statutes (primary), micro-F1, subset accuracy
Data: data_p5/tok/{split}.npz (tokens) + data_p5/clean/{split}.json ({id, labels}); test labels never stored.
"""
from __future__ import annotations

import json
import math
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import p2_peft as pp
import p4_cjpe as cj

NL = 100


class MSplit(cj.Split):
    """Tokens + multi-hot labels (labels = None for test)."""
    def __init__(self, npz, meta_json):
        z = np.load(npz)
        self.ids, self.off = z["ids"], z["offsets"]
        meta = json.load(open(meta_json))
        self.case_ids = [m["id"] for m in meta]
        self.n = len(meta)
        self.Y = None
        if meta[0]["labels"] is not None:
            self.Y = np.zeros((self.n, NL), dtype=np.float32)
            for i, m in enumerate(meta):
                self.Y[i, m["labels"]] = 1.0
        self.labels = self.Y                                         # (so cj helpers can be reused)


def build(ckpt, method="lora", bidirectional=True, partial_k=4, **kw):
    m = cj.build(ckpt, method, bidirectional, partial_k, **kw)      # DocClassifier with a 2-way head ...
    d = m.tagger.backbone.cfg.d_model
    head = nn.Linear(d, NL); nn.init.normal_(head.weight, 0.0, 0.02); nn.init.zeros_(head.bias)
    if isinstance(m.tagger.head, pp.DropPrefix):                     # ... replaced by a 100-way head
        m.tagger.head.head = head
    else:
        m.tagger.head = head
    return m


# ----------------------------------------------------------------------------- inference + metrics
@torch.no_grad()
def predict(model, split, mode="first", k=512, device="cuda", bs=32):
    """P(statute) for every case: (n, 100) float32."""
    model.eval(); out = np.zeros((split.n, NL), dtype=np.float32)
    order = np.argsort([min(k, split.off[i + 1] - split.off[i]) for i in range(split.n)])
    for b in range(0, split.n, bs):
        idx = order[b:b + bs]
        ids, mask = cj.collate([cj.section(split.arr(i), mode, k) for i in idx], device)
        with cj._autocast(device):
            out[idx] = torch.sigmoid(model(ids, mask).float()).cpu().numpy()
    return out


def decide(P, t):
    """Multi-hot decisions; t = scalar or (100,) thresholds; at least one label per case."""
    D = (P >= t).astype(np.int8)
    empty = D.sum(1) == 0
    D[empty, P[empty].argmax(1)] = 1
    return D


def metrics(Y, P, t=0.5):
    from sklearn.metrics import f1_score
    D = decide(P, t)
    return {"macro_f1": float(f1_score(Y, D, average="macro", zero_division=0)),
            "micro_f1": float(f1_score(Y, D, average="micro", zero_division=0)),
            "subset_acc": float((D == Y).all(1).mean()), "labels_per_case": float(D.sum(1).mean())}


def tune_global(Y, P):
    ts = np.linspace(0.05, 0.9, 35)
    f = [metrics(Y, P, t)["macro_f1"] for t in ts]
    return float(ts[int(np.argmax(f))])


def tune_per_label(Y, P, grid=np.linspace(0.05, 0.9, 35)):
    """Best threshold for each statute separately (maximises that statute's F1 on dev)."""
    T = np.zeros(NL)
    for j in range(NL):
        y, p = Y[:, j], P[:, j]
        best, bt = -1, 0.5
        for t in grid:
            d = p >= t; tp = (d & (y == 1)).sum(); fp = (d & (y == 0)).sum(); fn = (~d & (y == 1)).sum()
            f = 2 * tp / max(1, 2 * tp + fp + fn)
            if f > best:
                best, bt = f, t
        T[j] = bt
    return T


def crossfit_per_label(Y, P, seed=0):
    """Honest estimate of per-label thresholding: tune on one half of dev, score on the other, swap."""
    idx = np.random.RandomState(seed).permutation(len(Y)); a, b = idx[: len(Y) // 2], idx[len(Y) // 2:]
    D = np.zeros_like(Y, dtype=np.int8)
    D[b] = decide(P[b], tune_per_label(Y[a], P[a])); D[a] = decide(P[a], tune_per_label(Y[b], P[b]))
    from sklearn.metrics import f1_score
    return float(f1_score(Y, D, average="macro", zero_division=0))


# ----------------------------------------------------------------------------- training
def train(model, tr, dev, mode="first", k=512, epochs=1, lr=5e-4, head_lr=1e-3, bs=8, device="cuda",
          seed=0, warmup=0.03, evals_per_epoch=2, soft=None, kd_T=2.0, kd_lam=0.5,
          layerwise_unfreeze=False, log=print):
    """BCE with pos_weight (+ optional KD on teacher probabilities `soft`), AdamW, warm-up + linear decay,
    fp16. Keeps the evaluation with the best dev macro-F1 at the tuned GLOBAL threshold."""
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model.to(device)
    pos = tr.Y.sum(0); pw = torch.tensor(np.sqrt((tr.n - pos) / np.maximum(pos, 1)), dtype=torch.float, device=device)
    layers = list(pp._layers(model.tagger))
    if layerwise_unfreeze:
        for p in model.tagger.backbone.parameters():
            p.requires_grad = False
    head_ids = {id(p) for p in model.head().parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in head_ids
                                         and (p.requires_grad or layerwise_unfreeze)], "lr": lr},
                             {"params": list(model.head().parameters()), "lr": head_lr}], weight_decay=0.01)
    steps = epochs * math.ceil(tr.n / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))
    eval_at = {int(steps * (j + 1) / (epochs * evals_per_epoch)) for j in range(epochs * evals_per_epoch)}
    best, best_state, hist, step, t0, run = -1.0, None, [], 0, time.time(), 0.0
    log(f"trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.3f}M | {tr.n} cases | "
        f"{steps} steps | view {mode}-{k}" + (" | KD" if soft is not None else "")
        + (" | gradual unfreezing" if layerwise_unfreeze else ""))
    for ep in range(epochs):
        order = np.random.permutation(tr.n)
        for b in range(0, tr.n, bs):
            if layerwise_unfreeze:
                n_on = min(len(layers), 1 + int(step / max(1, steps) * len(layers)))
                for L in layers[len(layers) - n_on:]:
                    for p in L.parameters():
                        p.requires_grad = True
            model.train(); bi = order[b:b + bs]
            ids, mask = cj.collate([cj.section(tr.arr(i), mode, k) for i in bi], device)
            y = torch.from_numpy(tr.Y[bi]).to(device)
            with cj._autocast(device):
                z = model(ids, mask).float()
            loss = F.binary_cross_entropy_with_logits(z, y, pos_weight=pw)
            if soft is not None:                                      # KD on tempered sigmoids
                tp = torch.from_numpy(soft[bi]).to(device).float().clamp(1e-4, 1 - 1e-4)
                tz = torch.log(tp) - torch.log1p(-tp)                 # teacher logits
                kd = F.binary_cross_entropy_with_logits(z / kd_T, torch.sigmoid(tz / kd_T)) * kd_T * kd_T
                loss = (1 - kd_lam) * loss + kd_lam * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1; run += loss.item()
            if step % 500 == 0:
                log(f"   step {step}/{steps} | loss {run/500:.4f} | {time.time()-t0:.0f}s"); run = 0.0
            if step in eval_at:
                P = predict(model, dev, mode, k, device); t = tune_global(dev.Y, P); m = metrics(dev.Y, P, t)
                hist.append({"step": step, "macro_f1": m["macro_f1"], "micro_f1": m["micro_f1"], "t": t})
                log(f"   EVAL step {step}: dev macro-F1 {m['macro_f1']:.4f} | micro {m['micro_f1']:.4f} @t={t:.2f} "
                    f"| {m['labels_per_case']:.1f} labels/case | {time.time()-t0:.0f}s")
                if m["macro_f1"] > best:
                    best = m["macro_f1"]
                    best_state = {n: q.detach().cpu().clone() for n, q in model.named_parameters()
                                  if q.requires_grad or layerwise_unfreeze}
    model.load_state_dict({n: v.to(device) for n, v in best_state.items()}, strict=False)
    return hist, best


class ChunkPoolerML(cj.ChunkPooler):
    """Hierarchical multi-label head: BiGRU + attention over chunk embeddings -> 100 logits."""
    def __init__(self, d=768, h=256, dropout=0.2):
        super().__init__(d, h, dropout)
        self.out = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * h, NL))

Writing /kaggle/working/src/p5_lsi.py


## P5-2 · Background runner + smoke test

This is the same runner pattern as Phases 3 and 4: one detached process per GPU, so the runs survive browser reloads. Each run’s result JSON is written last and also serves as its “done” marker, so finished runs are skipped. Every run records:

dev macro- and micro-F1 at 0.5 and at the tuned global threshold;
the cross-fitted per-statute-threshold macro-F1;
F1 for each statute (for the rare-statute analysis);
dev and test probability matrices (n×100) for ensembling and the zips.

In [3]:
%%writefile /kaggle/working/src/p5_run.py
"""
p5_run.py -- background runner for Phase-5 (Legal Statute Identification, 100 labels), one process per GPU:

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p5_run.py --plan plan.json >> log 2>&1 &

plan.json = list of jobs. Job types:
  run  (default): {"name", "ckpt": "m1".."m4", "method", "mode": "first", "k": 512, "epochs", "lr",
                   "head_lr", "bs", "seed", "layerwise_unfreeze", "partial_k", method kwargs...,
                   KD: "teachers": [run names], "student_layers": [...], "kd_T", "kd_lam"}
  prob:  {"type": "prob", "name": X, "train": true}  re-load run X and add its TRAIN probabilities
         (needed when X is a KD teacher)
  hier:  {"type": "hier", "name": N, "base": X, "epochs": 10}  hierarchical model: encode every 512-token
         chunk of every case with fine-tuned run X (frozen), train a BiGRU + attention pooler over the
         chunk sequence with 100 sigmoid outputs; also reports where in the judgment the attention goes
Outputs per run: stats/p5_runs/<name>.json (written LAST = "done" marker), p5_ckpt/<name>.pt (trained
params), p5_prob/<name>.pkl ({"dev": (n,100) float16, "test": (n,100) float16[, "train"]}).
Model selection and thresholds use the CLEAN dev only; test labels do not exist on disk.
"""
import argparse
import json
import os
import pickle
import sys
import time
import traceback

import numpy as np
import torch
import torch.nn.functional as F

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p5_lsi as ls                                                   # noqa: E402
import p4_cjpe as cj                                                  # noqa: E402
import p2_peft as pp                                                  # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan", required=True)
PLAN = json.load(open(ap.parse_args().plan))
DEV = "cuda:0"                                                       # = the GPU given by CUDA_VISIBLE_DEVICES
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, PRB, CKD, CHK = (f"{WORK}/stats/p5_runs", f"{WORK}/p5_prob", f"{WORK}/p5_ckpt", f"{WORK}/p5_chunks")
for d in (OUT, PRB, CKD, CHK):
    os.makedirs(d, exist_ok=True)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


TR, DV, TE = (ls.MSplit(f"{WORK}/data_p5/tok/{s}.npz", f"{WORK}/data_p5/clean/{s}.json")
              for s in ("train", "dev", "test"))
log(f"data: train {TR.n} | clean dev {DV.n} | test {TE.n}")


def make_model(ckpt, method, student_layers=None, partial_k=4, **kw):
    m = ls.build(CK[ckpt], method, bidirectional=True, partial_k=partial_k, **kw)
    if student_layers:                                               # smaller KD student: keep a subset of layers
        pp.shrink_dense(m.tagger, tuple(student_layers))
    return m


def report(pd, pt):
    """Every dev number we need for one model: @0.5, @tuned global threshold, honest per-label
    thresholds (cross-fit), per-statute F1 (for the rare-statute analysis), test labels/case."""
    from sklearn.metrics import f1_score
    t = ls.tune_global(DV.Y, pd)
    per_label = f1_score(DV.Y, ls.decide(pd, t), average=None, zero_division=0)
    return t, {"dev@0.5": ls.metrics(DV.Y, pd, 0.5), "dev@tuned": ls.metrics(DV.Y, pd, t), "t": t,
               "dev_perlabel_crossfit_macro": ls.crossfit_per_label(DV.Y, pd),
               "dev_f1_per_statute": [round(float(v), 4) for v in per_label],
               "test_labels_per_case@tuned": float(ls.decide(pt, t).sum(1).mean())}


def run(name, ckpt="m1", method="lora", mode="first", k=512, epochs=1, lr=5e-4, head_lr=1e-3, bs=8, seed=0,
        layerwise_unfreeze=False, partial_k=4, teachers=None, student_layers=None, kd_T=2.0, kd_lam=0.5, **kw):
    t0 = time.time()
    model = make_model(ckpt, method, student_layers, partial_k, **kw)
    soft = None
    if teachers:                                                     # mean teacher probabilities on train cases
        soft = np.mean([pickle.load(open(f"{PRB}/{t}.pkl", "rb"))["train"].astype(np.float32) for t in teachers], 0)
    log(f"=== {name}: {ckpt} | {method} | {mode}-{k} | epochs {epochs} lr {lr}"
        + (f" | student layers {student_layers} | teachers {teachers}" if teachers or student_layers else ""))
    hist, best = ls.train(model, TR, DV, mode, k, epochs, lr, head_lr=head_lr, bs=bs, device=DEV, seed=seed,
                          soft=soft, kd_T=kd_T, kd_lam=kd_lam, layerwise_unfreeze=layerwise_unfreeze, log=log)
    pd = ls.predict(model, DV, mode, k, DEV)
    torch.cuda.synchronize(); ti = time.time()
    pt = ls.predict(model, TE, mode, k, DEV)
    torch.cuda.synchronize(); infer = round(time.time() - ti, 1)
    t, rep = report(pd, pt)
    out = {"cfg": dict(ckpt=ckpt, method=method, mode=mode, k=k, epochs=epochs, lr=lr, head_lr=head_lr, bs=bs,
                       seed=seed, layerwise_unfreeze=layerwise_unfreeze, partial_k=partial_k, teachers=teachers,
                       student_layers=student_layers, kd_T=kd_T, kd_lam=kd_lam, **kw),
           "trainable_M": round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3),
           "total_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
           "infer_test_s": infer, "hist": hist, "min": round((time.time() - t0) / 60, 1), **rep}
    torch.save({n: p.detach().half().cpu() for n, p in model.named_parameters()
                if p.requires_grad or layerwise_unfreeze or method in ("full", "partial")}, f"{CKD}/{name}.pt")
    pickle.dump({"dev": pd.astype(np.float16), "test": pt.astype(np.float16)}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)        # written LAST = "done" marker
    log(f"DONE {name}: dev macro-F1 {out['dev@tuned']['macro_f1']:.4f} @t={t:.2f} (micro {out['dev@tuned']['micro_f1']:.4f}) "
        f"| per-label thr (cross-fit) {out['dev_perlabel_crossfit_macro']:.4f} | {out['min']} min")
    del model; torch.cuda.empty_cache()


def _load_run(name):
    """Rebuild run `name` from its result file and load its trained weights."""
    R = json.load(open(f"{OUT}/{name}.json")); c = {k: v for k, v in R["cfg"].items() if v is not None}
    mode, k = c.pop("mode"), c.pop("k")
    for key in ("epochs", "lr", "head_lr", "bs", "seed", "layerwise_unfreeze", "teachers", "kd_T", "kd_lam"):
        c.pop(key, None)
    m = make_model(c.pop("ckpt"), c.pop("method"), c.pop("student_layers", None), c.pop("partial_k", 4), **c)
    sd = {n: v.float() for n, v in torch.load(f"{CKD}/{name}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False)
    return m.to(DEV), mode, k


def prob_job(name, train=False):
    m, mode, k = _load_run(name)
    path = f"{PRB}/{name}.pkl"; out = pickle.load(open(path, "rb"))
    if train and "train" not in out:
        ti = time.time(); out["train"] = ls.predict(m, TR, mode, k, DEV).astype(np.float16)
        log(f"   {name}: train probs in {time.time()-ti:.0f}s")
    pickle.dump(out, open(path, "wb"))
    chk = ls.metrics(DV.Y, ls.predict(m, DV, mode, k, DEV), 0.5)["macro_f1"]
    log(f"PROB {name}: dev re-check {chk:.4f} (should equal its dev@0.5 score)")
    del m; torch.cuda.empty_cache()


def _pad(E, idx):
    C = max(len(E[i]) for i in idx)
    x = np.zeros((len(idx), C, E[idx[0]].shape[1]), dtype=np.float32); mk = np.zeros((len(idx), C), dtype=bool)
    for b, i in enumerate(idx):
        x[b, :len(E[i])] = E[i]; mk[b, :len(E[i])] = True
    return torch.from_numpy(x).to(DEV), torch.from_numpy(mk).to(DEV)


@torch.no_grad()
def _pool_predict(pool, E, bs=64, return_att=False):
    pool.eval(); n = len(E); p = np.zeros((n, ls.NL), dtype=np.float32); atts = [None] * n
    order = np.argsort([len(e) for e in E])
    for b in range(0, n, bs):
        idx = order[b:b + bs]; x, mk = _pad(E, idx)
        o, a = pool(x, mk, return_att=True)
        p[idx] = torch.sigmoid(o).cpu().numpy()
        if return_att:
            for j, i in enumerate(idx):
                atts[i] = a[j, :len(E[i])].cpu().numpy()
    return (p, atts) if return_att else p


def hier_job(name, base, epochs=10, lr=1e-3, chunk=512, max_chunks=48, bs=32, seed=0):
    t0 = time.time(); torch.manual_seed(seed); np.random.seed(seed)
    cache = f"{CHK}/{base}.pkl"
    if os.path.exists(cache):                                        # chunk embeddings survive a session stop
        E = pickle.load(open(cache, "rb")); log(f"   loaded chunk embeddings of {base}")
    else:
        m, _, _ = _load_run(base); E = {}
        for s, sp in (("dev", DV), ("test", TE), ("train", TR)):
            ti = time.time(); E[s] = cj.encode_chunks(m, sp, chunk, DEV, max_chunks=max_chunks)
            log(f"   encoded {s}: {sum(len(e) for e in E[s])} chunks in {time.time()-ti:.0f}s")
        del m; torch.cuda.empty_cache(); pickle.dump(E, open(cache, "wb"))
    pool = ls.ChunkPoolerML(E["train"][0].shape[1]).to(DEV)
    pos = TR.Y.sum(0); pw = torch.tensor(np.sqrt((TR.n - pos) / np.maximum(pos, 1)), dtype=torch.float, device=DEV)
    opt = torch.optim.AdamW(pool.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(TR.n / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    log(f"=== {name}: hierarchical BiGRU+attention over chunks of {base} | {sum(p.numel() for p in pool.parameters())/1e6:.2f}M params")
    best, best_state, hist = -1.0, None, []
    for ep in range(epochs):
        pool.train(); order = np.random.permutation(TR.n); tot = 0.0
        for b in range(0, TR.n, bs):
            idx = order[b:b + bs]; x, mk = _pad(E["train"], idx)
            loss = F.binary_cross_entropy_with_logits(pool(x, mk), torch.from_numpy(TR.Y[idx]).to(DEV), pos_weight=pw)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(pool.parameters(), 1.0); opt.step(); sched.step(); tot += loss.item()
        pd = _pool_predict(pool, E["dev"]); t = ls.tune_global(DV.Y, pd); mt = ls.metrics(DV.Y, pd, t)
        hist.append({"epoch": ep + 1, "macro_f1": mt["macro_f1"], "micro_f1": mt["micro_f1"], "t": t})
        log(f"   epoch {ep+1}: loss {tot/np.ceil(TR.n/bs):.4f} | dev macro-F1 {mt['macro_f1']:.4f} "
            f"(micro {mt['micro_f1']:.4f}) @t={t:.2f}")
        if mt["macro_f1"] > best:
            best = mt["macro_f1"]; best_state = {k: v.detach().clone() for k, v in pool.state_dict().items()}
    pool.load_state_dict(best_state)
    pd, att = _pool_predict(pool, E["dev"], return_att=True); pt = _pool_predict(pool, E["test"])
    t, rep = report(pd, pt)
    prof, n_docs = np.zeros(10), 0                                   # mean attention mass per position decile
    for a in att:
        if len(a) >= 3:                                              # (cases with >= 3 chunks)
            p_ = np.minimum(9, (np.arange(len(a)) / len(a) * 10).astype(int))
            np.add.at(prof, p_, a); n_docs += 1
    out = {"cfg": {"type": "hier", "base": base, "epochs": epochs, "lr": lr, "chunk": chunk, "max_chunks": max_chunks},
           "trainable_M": round(sum(p.numel() for p in pool.parameters()) / 1e6, 3), "total_M": 0.0,
           "infer_test_s": 0.0, "hist": hist, "min": round((time.time() - t0) / 60, 1), **rep,
           "attention_by_position_decile": [round(float(v), 4) for v in prof / max(1, n_docs)]}
    torch.save(pool.state_dict(), f"{CKD}/{name}.pt")
    pickle.dump({"dev": pd.astype(np.float16), "test": pt.astype(np.float16)}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump(out, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name}: dev macro-F1 {out['dev@tuned']['macro_f1']:.4f} @t={t:.2f} | per-label (cross-fit) "
        f"{out['dev_perlabel_crossfit_macro']:.4f} | attention by decile {out['attention_by_position_decile']} | {out['min']} min")


for job in PLAN:
    job = dict(job); name = job.pop("name"); typ = job.pop("type", "run")
    try:
        if typ == "prob":
            prob_job(name, job.get("train", False))
        elif os.path.exists(f"{OUT}/{name}.json"):
            log(f"skip {name} (already done)")
        elif typ == "hier":
            hier_job(name, **job)
        else:
            run(name, **job)
    except Exception:
        log(f"!!! {name} FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Writing /kaggle/working/src/p5_run.py


In [4]:
# ---- P5-2 smoke test: imports, splits, label matrix, a 100-way M1 forward pass on 2 cases (CPU) ----
import sys, importlib, numpy as np, torch
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p5_lsi as ls; importlib.reload(ls)                            # reload in case of an older copy

TR = ls.MSplit(f"{WORK}/data_p5/tok/train.npz", f"{WORK}/data_p5/clean/train.json")
TE = ls.MSplit(f"{WORK}/data_p5/tok/test.npz", f"{WORK}/data_p5/clean/test.json")
print("train", TR.n, "| Y", TR.Y.shape, "| labels/case", TR.Y.sum(1).mean().round(2),
      "| test labels stored:", TE.Y is not None)                     # must print False

m = ls.build(f"{WORK}/checkpoints/m1_dense_100m/best.pt", "lora")    # 100-way head on M1 + LoRA
ids, mask = ls.cj.collate([ls.cj.section(TR.arr(i), "first", 64) for i in range(2)], "cpu")
with torch.no_grad():
    print("logits", tuple(m(ids, mask).shape), "| trainable",
          round(sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6, 3), "M")   # expect (2, 100)
P = np.random.rand(50, 100); Y = (np.random.rand(50, 100) > 0.95).astype(float)
print("metrics ok:", ls.metrics(Y, P, ls.tune_global(Y, P))["macro_f1"] >= 0)

train 42102 | Y (42102, 100) | labels/case 3.7 | test labels stored: False
logits (2, 100) | trainable 0.323 M
metrics ok: True


## P5-3 · Section study (which part of the judgment carries the statutes?)

All runs use M1 + LoRA for 1 epoch, comparing the first, last, middle and head+tail 512-token windows, plus first-1024 to test whether length matters. The work is split into two background queues, one per GPU, with roughly equal load (about 60–70 min in total). The winning view becomes the default for the technique comparison.

In [1]:
# ---- P5-3L: launch the section study in the background (one process per GPU) ----
import json, os, subprocess
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/stats", exist_ok=True)

base = dict(ckpt="m1", method="lora", epochs=1, lr=5e-4, k=512)     # same recipe; only the view changes
plans = {0: [dict(base, name="m1_lora_first512", mode="first"),
             dict(base, name="m1_lora_headtail512", mode="headtail"),
             dict(base, name="m1_lora_middle512", mode="middle")],
         1: [dict(base, name="m1_lora_last512", mode="last"),
             dict(base, name="m1_lora_first1024", mode="first", k=1024)]}   # 1024 costs ~2x -> balanced

for g, plan in plans.items():
    pf = f"{WORK}/stats/p5_plan_gpu{g}.json"
    busy = subprocess.run(f"pgrep -f '[p]5_run.py --plan {pf}'", shell=True, capture_output=True).stdout
    if busy:                                                          # never start a 2nd copy on the same GPU
        print(f"GPU {g}: already running, not relaunched"); continue
    json.dump(plan, open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                   f"python p5_run.py --plan {pf} >> {WORK}/stats/p5_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: launched {[p['name'] for p in plan]}")

GPU 0: launched ['m1_lora_first512', 'm1_lora_headtail512', 'm1_lora_middle512']
GPU 1: launched ['m1_lora_last512', 'm1_lora_first1024']


In [2]:
# ---- P5-W: what is running, last log lines, finished-run table ----
import glob, json, os, subprocess
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout
print(sh("pgrep -af '[p]5_run.py' | cut -c1-140") or "no p5_run.py process running")
print(sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader"))
for g in (0, 1):
    f = f"{WORK}/stats/p5_gpu{g}.log"
    if os.path.exists(f):
        print(f"--- GPU {g} log (UTC) | failures so far: {sh(f'grep -c \"!!!\" {f}').strip()} ---")
        print(sh(f"tail -n 6 {f}"))

rows = []
for f in glob.glob(f"{WORK}/stats/p5_runs/*.json"):
    R = json.load(open(f)); c = R["cfg"]
    view = f"hier({c['base']})" if c.get("type") == "hier" else f"{c['mode']}-{c['k']}"
    rows.append((R["dev@tuned"]["macro_f1"], os.path.basename(f)[:-5], view, R["dev@tuned"]["micro_f1"],
                 R["dev_perlabel_crossfit_macro"], R["t"], R["dev@tuned"]["labels_per_case"], R["trainable_M"], R["min"]))
print(f"{'run':30s} {'view':22s} {'macroF1':>8s} {'micro':>7s} {'perlab':>7s} {'t':>5s} {'lab/c':>6s} {'trainM':>7s} {'min':>5s}")
for r in sorted(rows, reverse=True):
    print(f"{r[1]:30s} {r[2]:22s} {r[0]:8.4f} {r[3]:7.4f} {r[4]:7.4f} {r[5]:5.2f} {r[6]:6.2f} {r[7]:7.3f} {r[8]:5.1f}")

126 python3 p5_run.py --plan /kaggle/working/stats/p5_plan5_gpu0.json
128 python3 p5_run.py --plan /kaggle/working/stats/p5_plan5_gpu1.json

0, 99 %, 6825 MiB
1, 100 %, 1565 MiB

--- GPU 0 log (UTC) | failures so far: 3 ---
18:31:33    EVAL step 2631: dev macro-F1 0.3273 | micro 0.4129 @t=0.53 | 5.6 labels/case | 1667s
18:35:08    step 3000/10526 | loss 0.2830 | 1883s
18:40:00    step 3500/10526 | loss 0.2823 | 2174s
20:21:40 data: train 42102 | clean dev 10085 | test 13019
20:21:43 === m1_full_first1024: m1 | full | first-1024 | epochs 2 lr 3e-05
20:21:46 trainable 100.171M | 42102 cases | 10526 steps | view first-1024

--- GPU 1 log (UTC) | failures so far: 2 ---
18:03:39 data: train 42102 | clean dev 10085 | test 13019
18:03:39 skip m1_partial4_first1024 (already done)
18:12:22    encoded dev: 24206 chunks in 518s
18:23:24    encoded test: 30121 chunks in 662s
20:21:40 data: train 42102 | clean dev 10085 | test 13019
20:21:40 skip m1_partial4_first1024 (already done)

run           

In [2]:
# ---- P5-B: extend the backup allow-list + upload Phase-5 work now ----
import glob, os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
WORK, REPO = "/kaggle/working", "Utkarsh-1718-1/cs781-phase1-work"
P5 = ["data_p5/tok/**", "data_p5/clean/**", "p5_ckpt/**", "p5_prob/**", "p5_chunks/**", "submissions_p5/**"]

# 1) make the regular backup script include Phase 5 from now on
for f in glob.glob(f"{WORK}/**/backup_hf.py", recursive=True):
    s = open(f).read()
    if '"p5_ckpt/**"' not in s and '"submissions_p2/**",' in s:
        s = s.replace('"submissions_p2/**",', '"submissions_p2/**", ' + ", ".join(f'"{p}"' for p in P5) + ",", 1)
        open(f, "w").write(s); print("allow-list extended:", f)

# 2) upload Phase-5 data/results + code + stats (repo type detected, token from the Kaggle secret only)
api = HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))
rt = None
for t in ("model", "dataset"):                                         # find which repo type it is
    try:
        api.repo_info(REPO, repo_type=t); rt = t; break
    except Exception:
        pass
api.upload_folder(repo_id=REPO, repo_type=rt, folder_path=WORK, commit_message="phase 5 backup",
                  allow_patterns=P5 + ["src/**", "stats/**"], ignore_patterns=["**/__pycache__/**"])
print("backup done →", REPO, f"({rt})")

allow-list extended: /kaggle/working/src/backup_hf.py


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

backup done → Utkarsh-1718-1/cs781-phase1-work (model)


## P5-4L · Techniques × architectures (+ first hierarchical model)

These runs use the same recipe for every technique and architecture: the first-512 view, 1 epoch, and the Phase 4 learning rates. The positions are statistically equal, so first-512 is the cheapest fair choice. The first hierarchical model runs right after full fine-tuning: it encodes every 512-token chunk with the fine-tuned model, then a BiGRU + attention pooler reads the whole judgment. This directly tests the “length matters” finding.

In [1]:
# ---- P5-4L: technique x architecture comparison, two background queues (~3.5 h each) ----
import json, os, subprocess
WORK = "/kaggle/working"
B = dict(mode="first", k=512, epochs=1)                              # identical view/epochs for every run

plans = {
    0: [dict(name="m1_full_first512",      ckpt="m1", method="full",       lr=3e-5, **B),
        dict(type="hier", name="m1_full512_hier", base="m1_full_first512", chunk=512, max_chunks=48, epochs=10),
        dict(name="m1_unfreeze_first512",  ckpt="m1", method="full",       lr=5e-5, layerwise_unfreeze=True, **B),
        dict(name="m1_rslora_first512",    ckpt="m1", method="rslora",     lr=2e-4, rs_r=16, **B),
        dict(name="m1_ia3_first512",       ckpt="m1", method="ia3",        lr=3e-3, **B),
        dict(name="m3_full_first512",      ckpt="m3", method="full",       lr=5e-5, **B)],
    1: [dict(name="m1_partial4_first512",  ckpt="m1", method="partial",    lr=1e-4, partial_k=4, **B),
        dict(name="m1_lorma_plus_first512", ckpt="m1", method="lorma_plus", lr=5e-4, **B),
        dict(name="m1_adapter_first512",   ckpt="m1", method="adapter",    lr=5e-4, **B),
        dict(name="m1_prompt_first512",    ckpt="m1", method="prompt",     lr=1e-2, **B),
        dict(name="m2_lorma_plus_first512", ckpt="m2", method="lorma_plus", lr=5e-4, **B),
        dict(name="m4_adapter_first512",   ckpt="m4", method="adapter",    lr=5e-4, **B),
        dict(name="m1_normgain_first512",  ckpt="m1", method="normgain",   lr=1e-3, **B)],
}

import torch; assert torch.cuda.is_available(), "GPU not attached - turn on the T4x2 accelerator first"
for g, plan in plans.items():
    pf = f"{WORK}/stats/p5_plan4_gpu{g}.json"
    if subprocess.run("pgrep -f '[p]5_run.py'", shell=True, capture_output=True).stdout and \
       subprocess.run(f"pgrep -f '[p]5_run.py --plan {WORK}/stats/p5_plan.*gpu{g}'", shell=True, capture_output=True).stdout:
        print(f"GPU {g}: a p5 plan is still running, not relaunched"); continue   # never 2 processes per GPU
    json.dump(plan, open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                   f"python p5_run.py --plan {pf} >> {WORK}/stats/p5_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: launched {[p['name'] for p in plan]}")

GPU 0: launched ['m1_full_first512', 'm1_full512_hier', 'm1_unfreeze_first512', 'm1_rslora_first512', 'm1_ia3_first512', 'm3_full_first512']
GPU 1: launched ['m1_partial4_first512', 'm1_lorma_plus_first512', 'm1_adapter_first512', 'm1_prompt_first512', 'm2_lorma_plus_first512', 'm4_adapter_first512', 'm1_normgain_first512']


In [3]:
# ---- P5-R: what survived the session stop? ----
import glob, os, subprocess, torch
WORK = "/kaggle/working"
need = {"code (src/p5_run.py)": f"{WORK}/src/p5_run.py",
        "Phase-5 tokens": f"{WORK}/data_p5/tok/train.npz",
        "Phase-5 labels": f"{WORK}/data_p5/clean/train.json",
        "M1 checkpoint": f"{WORK}/checkpoints/m1_dense_100m/best.pt"}
for k, p in need.items():
    print(f"{'OK     ' if os.path.exists(p) else 'MISSING'} {k}")
print("GPU attached:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
print("p5 processes:", subprocess.run("pgrep -af '[p]5_run.py'", shell=True, capture_output=True, text=True).stdout or "none")
done = sorted(os.path.basename(f)[:-5] for f in glob.glob(f"{WORK}/stats/p5_runs/*.json"))
print(f"finished p5 runs ({len(done)}):", done)
# if anything is MISSING -> run your R1 restore cell (HF snapshot_download) first, then P5-4L

OK      code (src/p5_run.py)
OK      Phase-5 tokens
OK      Phase-5 labels
OK      M1 checkpoint
GPU attached: True | GPUs: 2
p5 processes: 112 python3 p5_run.py --plan /kaggle/working/stats/p5_plan4_gpu0.json
119 python3 p5_run.py --plan /kaggle/working/stats/p5_plan4_gpu1.json

finished p5 runs (7): ['m1_full_first512', 'm1_lora_first1024', 'm1_lora_first512', 'm1_lora_headtail512', 'm1_lora_last512', 'm1_lora_middle512', 'm1_partial4_first512']


## P5-5 · Final models: full and partial fine-tunes on first-1024, each with a hierarchical model on top

The final models combine the three findings: full fine-tuning is best, longer input helps, and the hierarchical model helps most. Each base trains for 2 epochs on first-1024. Each base then gets a hierarchical model that reads the whole judgment in 1024-token chunks. Running the cell launches a queue on every GPU that is free: run it now to start GPU 0, and again after 06:05 to start GPU 1.

In [1]:
# ---- P5-5: final models; launches only on GPUs with no p5 process running (re-run to fill the other GPU) ----
import json, os, subprocess, torch
WORK = "/kaggle/working"
assert torch.cuda.is_available(), "GPU not attached - turn on the T4x2 accelerator first"
F = dict(mode="first", k=1024, epochs=2, bs=8)                      # best view (+length) and 2 epochs

plans = {
    0: [dict(name="m1_full_first1024",     ckpt="m1", method="full",    lr=3e-5, **F),          # ~2.2 h
        dict(type="hier", name="m1_full1024_hier", base="m1_full_first1024",
             chunk=1024, max_chunks=24, epochs=10)],                                          # ~30 min
    1: [dict(name="m1_partial4_first1024", ckpt="m1", method="partial", lr=1e-4, partial_k=4, **F),  # ~1.5 h
        dict(type="hier", name="m1_partial1024_hier", base="m1_partial4_first1024",
             chunk=1024, max_chunks=24, epochs=10)],                                          # ~30 min
}
running = subprocess.run("pgrep -af '[p]5_run.py'", shell=True, capture_output=True, text=True).stdout
for g, plan in plans.items():
    if f"gpu{g}.json" in running:                                     # this GPU still has a queue -> leave it
        print(f"GPU {g}: busy, run this cell again when it is free"); continue
    pf = f"{WORK}/stats/p5_plan5_gpu{g}.json"
    json.dump(plan, open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={g} PYTHONUNBUFFERED=1 setsid nohup "
                   f"python p5_run.py --plan {pf} >> {WORK}/stats/p5_gpu{g}.log 2>&1 < /dev/null &", shell=True)
    print(f"GPU {g}: launched {[p['name'] for p in plan]}")

GPU 0: launched ['m1_full_first1024', 'm1_full1024_hier']
GPU 1: launched ['m1_partial4_first1024', 'm1_partial1024_hier']


In [5]:
# ---- P5-B2: wait for the P5-4 queue to finish -> backup to HF -> "SAFE TO STOP" ----
import glob, os, subprocess, time
from IPython.display import clear_output
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
WORK, REPO = "/kaggle/working", "Utkarsh-1718-1/cs781-phase1-work"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

# 1) wait (and keep the session busy) while any P5-4 comparison run is still training
t0 = time.time()
while sh("pgrep -f '[p]5_run.py --plan .*p5_plan4'") and time.time() - t0 < 90 * 60:   # safety cap 90 min
    clear_output(wait=True)
    ist = time.strftime("%H:%M", time.gmtime(time.time() + 5.5 * 3600))
    print(f"[{ist} IST] waiting for the comparison queue... GPU 1: {sh(f'tail -n 1 {WORK}/stats/p5_gpu1.log')[:140]}")
    time.sleep(60)

# 2) upload all Phase-5 work (only new/changed files are actually transferred)
api = HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))      # token from the Kaggle secret only
api.upload_folder(repo_id=REPO, repo_type="model", folder_path=WORK, commit_message="phase 5 backup (P5-4 done)",
                  allow_patterns=["data_p5/tok/**", "data_p5/clean/**", "p5_ckpt/**", "p5_prob/**",
                                  "p5_chunks/**", "submissions_p5/**", "src/**", "stats/**"],
                  ignore_patterns=["**/__pycache__/**"])

# 3) report what is saved + whether anything is still training
done = sorted(os.path.basename(f)[:-5] for f in glob.glob(f"{WORK}/stats/p5_runs/*.json"))
print(f"backed up {len(done)} finished Phase-5 runs:", done)
left = sh("pgrep -af '[p]5_run.py'")
print("\nSAFE TO STOP ✅" if not left else
      f"\nbackup done, but still training (will restart from scratch next session):\n{left}")

[06:02 IST] waiting for the comparison queue... GPU 1: 00:29:45    EVAL step 5263: dev macro-F1 0.2948 | micro 0.4073 @t=0.50 | 5.1 labels/case | 1309s


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

backed up 18 finished Phase-5 runs: ['m1_adapter_first512', 'm1_full512_hier', 'm1_full_first512', 'm1_ia3_first512', 'm1_lora_first1024', 'm1_lora_first512', 'm1_lora_headtail512', 'm1_lora_last512', 'm1_lora_middle512', 'm1_lorma_plus_first512', 'm1_normgain_first512', 'm1_partial4_first512', 'm1_prompt_first512', 'm1_rslora_first512', 'm1_unfreeze_first512', 'm2_lorma_plus_first512', 'm3_full_first512', 'm4_adapter_first512']

backup done, but still training (will restart from scratch next session):
164 python3 p5_run.py --plan /kaggle/working/stats/p5_plan5_gpu0.json


## P5-Q · Queue manager (P5-5 + P5-6: seeds, KD), keep-alive, backup

Stages run in order on each GPU and start when that GPU is free and their inputs exist:

GPU 1: finish P5-5 → 2 extra pooler seeds for the partial hierarchical model + a 6-layer student without KD (control) → teacher train-probabilities + the 6-layer student with KD (it waits for the full model).
GPU 0: finish P5-5 → 2 extra pooler seeds for the full hierarchical model.

Extra seeds are cheap, since the chunk embeddings are cached, and they give the ensemble diversity. KD vs no-KD with the same 62M student isolates the effect of distillation. Everything is backed up to HF at the end.

In [3]:
# ---- P5-Q: queue manager = keep-alive + launches every remaining GPU stage when its GPU is free ----
import json, os, subprocess, time
from IPython.display import clear_output
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
done = lambda n: os.path.exists(f"{WORK}/stats/p5_runs/{n}.json")          # result file = "finished"
busy = lambda g: bool(sh(f"pgrep -f '[p]5_run.py --plan .*gpu{g}.json'"))   # a queue is running on GPU g

F = dict(mode="first", k=1024, epochs=2, bs=8)                               # P5-5 recipe (unchanged)
STU = dict(ckpt="m1", method="full", mode="first", k=1024, epochs=1, lr=5e-5, bs=8,
           student_layers=[0, 2, 4, 6, 8, 10])                               # 6-layer student (62M params)
H = dict(type="hier", chunk=1024, max_chunks=24, epochs=10)                  # hierarchical pooler settings

# stages run in list order per GPU; a stage starts when its GPU is free AND the runs in "needs" exist
STAGES = [
    dict(tag="5a", gpu=0, needs=[], plan=[                                   # = P5-5 GPU 0 (skips if done)
        dict(name="m1_full_first1024", ckpt="m1", method="full", lr=3e-5, **F),
        dict(name="m1_full1024_hier", base="m1_full_first1024", **H)]),
    dict(tag="5b", gpu=1, needs=[], plan=[                                   # = P5-5 GPU 1 (skips if done)
        dict(name="m1_partial4_first1024", ckpt="m1", method="partial", lr=1e-4, partial_k=4, **F),
        dict(name="m1_partial1024_hier", base="m1_partial4_first1024", **H)]),
    dict(tag="6a", gpu=1, needs=[], plan=[                                   # extra pooler seeds (cached chunks)
        dict(name="m1_partial1024_hier_s1", base="m1_partial4_first1024", seed=1, **H),
        dict(name="m1_partial1024_hier_s2", base="m1_partial4_first1024", seed=2, **H),
        dict(name="nokd_student_m1x6_first1024", **STU)]),                   # KD control: same student, no teacher
    dict(tag="6b", gpu=1, needs=["m1_full_first1024"], plan=[                # KD needs the teacher's TRAIN probs
        dict(type="prob", name="m1_full_first1024", train=True),
        dict(name="kd_student_m1x6_first1024", teachers=["m1_full_first1024"], kd_T=2.0, kd_lam=0.5, **STU)]),
    dict(tag="6c", gpu=0, needs=[], plan=[
        dict(name="m1_full1024_hier_s1", base="m1_full_first1024", seed=1, **H),
        dict(name="m1_full1024_hier_s2", base="m1_full_first1024", seed=2, **H)]),
]
names = lambda st: [j["name"] for j in st["plan"] if j.get("type") != "prob"]
complete = lambda st: all(done(n) for n in names(st))
tries = {st["tag"]: 0 for st in STAGES}

def launch(st):
    pf = f"{WORK}/stats/p5_plan{st['tag']}_gpu{st['gpu']}.json"
    json.dump(st["plan"], open(pf, "w"), indent=1)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES={st['gpu']} PYTHONUNBUFFERED=1 setsid nohup "
                   f"python p5_run.py --plan {pf} >> {WORK}/stats/p5_gpu{st['gpu']}.log 2>&1 < /dev/null &", shell=True)
    tries[st["tag"]] += 1

t0, log = time.time(), []
while time.time() - t0 < 9 * 3600:                                           # safety cap 9 h
    pending = [st for st in STAGES if not complete(st) and tries[st["tag"]] < 2]   # max 2 attempts per stage
    launched = False
    for g in (0, 1):
        if busy(g):
            continue
        nxt = next((st for st in pending if st["gpu"] == g), None)           # next unfinished stage of this GPU
        if nxt and all(done(n) for n in nxt["needs"]):
            launch(nxt); launched = True
            log.append(f"{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST: launched {nxt['tag']} on GPU {g}")
    if not launched and not busy(0) and not busy(1):
        break                                                                # nothing running and nothing startable
    clear_output(wait=True)
    print(f"[{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST]  stages: " +
          " | ".join(f"{st['tag']}:{'done' if complete(st) else ('run' if tries[st['tag']] else 'wait')}" for st in STAGES))
    for g in (0, 1):
        print(f"GPU {g}: {sh(f'tail -n 1 {WORK}/stats/p5_gpu{g}.log')[:150]}")
    print("\n".join(log[-6:]))
    time.sleep(60)

# ---- everything finished -> backup to HF so a session stop cannot lose it ----
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 5 backup (final models)", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["p5_ckpt/**", "p5_prob/**", "p5_chunks/**", "src/**", "stats/**"])
print("\nall stages:", {st["tag"]: ("done" if complete(st) else "NOT done") for st in STAGES}, "| backup done")

[04:58 IST]  stages: 5a:done | 5b:done | 6a:done | 6b:done | 6c:run
GPU 0: 23:28:28    epoch 10: loss 0.1322 | dev macro-F1 0.4103 (micro 0.5029) @t=0.62
GPU 1: 23:11:11 PLAN DONE
02:47 IST: launched 6a on GPU 1
03:57 IST: launched 6b on GPU 1
04:54 IST: launched 6c on GPU 0


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


all stages: {'5a': 'done', '5b': 'done', '6a': 'done', '6b': 'done', '6c': 'done'} | backup done


## P5-7 · Ensembles + threshold rules

Everything is chosen on the clean dev only. The cell builds three ensembles:

greedy forward selection (with replacement, so better models can count more than once);
top-5 single models;
all hierarchical models.

Each ensemble is scored with three decision rules: one global threshold, one threshold per statute, and per-statute thresholds shrunk halfway toward the global one (against overfitting). Every score is honest: the members and thresholds are chosen on one half of dev and scored on the other half, then the halves are swapped. Single models get the global rule, which gives the technique table its test scores.

In [4]:
# ---- P5-7: ensembles + threshold rules, chosen on the CLEAN dev only (honest 2-fold estimates) ----
import sys, glob, json, os, pickle, numpy as np
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p5_lsi as ls
Y = ls.MSplit(f"{WORK}/data_p5/tok/dev.npz", f"{WORK}/data_p5/clean/dev.json").Y   # (10085, 100) multi-hot

# 1) every finished model's dev + test probabilities
P = {}
for f in sorted(glob.glob(f"{WORK}/p5_prob/*.pkl")):
    d = pickle.load(open(f, "rb"))
    if "dev" in d and "test" in d:
        P[os.path.basename(f)[:-4]] = {s: d[s].astype(np.float32) for s in ("dev", "test")}
print(len(P), "models:", sorted(P))

# 2) fast macro-F1 + threshold rules (global / one per statute / per-statute shrunk halfway to global)
TS = np.linspace(0.05, 0.9, 35)
def mf1(Yt, D):
    D = D.astype(np.float32); tp = (D * Yt).sum(0); fp = (D * (1 - Yt)).sum(0); fn = ((1 - D) * Yt).sum(0)
    return float(np.mean(2 * tp / np.maximum(2 * tp + fp + fn, 1)))
def tune_t(Yt, Pm):
    f = [mf1(Yt, ls.decide(Pm, t)) for t in TS]; i = int(np.argmax(f)); return f[i], float(TS[i])
RULES = ("global", "perlabel", "shrunk")
def thresholds(Yt, Pm, rule):
    t = tune_t(Yt, Pm)[1]
    if rule == "global":
        return np.full(ls.NL, t)
    T = ls.tune_per_label(Yt, Pm)
    return T if rule == "perlabel" else 0.5 * (T + t)

# 3) greedy forward selection (with replacement = learned weights) on a subset of dev
def greedy(idx, max_steps=10):
    sel, best, s = [], -1.0, None
    for _ in range(max_steps):
        f, n = max((tune_t(Y[idx], P[m]["dev"][idx] if s is None else (s + P[m]["dev"][idx]) / (len(sel) + 1))[0], m)
                   for m in P)
        if f <= best + 1e-4:
            break
        sel.append(n); best = f; s = P[n]["dev"][idx] if s is None else s + P[n]["dev"][idx]
    return sel

# 4) honest score: choose members + thresholds on one half of dev, score on the other half, swap, average
perm = np.random.RandomState(0).permutation(len(Y)); h = len(Y) // 2
HALVES = [(perm[:h], perm[h:]), (perm[h:], perm[:h])]
def honest(members_fn):
    out = {r: [] for r in RULES}
    for a, b in HALVES:
        Pd = np.mean([P[m]["dev"] for m in members_fn(a)], 0)
        for r in RULES:
            out[r].append(mf1(Y[b], ls.decide(Pd[b], thresholds(Y[a], Pd[a], r))))
    return {r: float(np.mean(v)) for r, v in out.items()}

SYS = {}
def add(name, members_fn, rules=RULES):
    mem = members_fn(np.arange(len(Y))); hs = honest(members_fn)
    Pd = np.mean([P[m]["dev"] for m in mem], 0)
    for r in rules:
        T = thresholds(Y, Pd, r)
        SYS[f"{name}_{r}" if len(rules) > 1 else name] = {
            "members": mem, "rule": r, "T": [float(x) for x in T],
            "dev_insample": mf1(Y, ls.decide(Pd, T)), "dev_honest": hs[r]}
    print(f"{name:34s} honest " + " | ".join(f"{r} {hs[r]:.4f}" for r in rules) + f"  <- {len(mem)} members")

top5 = lambda idx: sorted(P, key=lambda m: tune_t(Y[idx], P[m]["dev"][idx])[0], reverse=True)[:5]
hier = [m for m in P if "_hier" in m]
add("ens_greedy", greedy)                                                    # main candidate
add("ens_top5", top5)
if hier:
    add("ens_hier", lambda idx: hier)
for m in P:                                                                  # every single model (report table)
    add(m, lambda idx, m=m: [m], rules=("global",))

json.dump(SYS, open(f"{WORK}/stats/p5_systems.json", "w"), indent=1)
print(f"\n{'system':40s}{'honest':>8s}{'in-sample':>11s}  members")
for k in sorted(SYS, key=lambda k: -SYS[k]["dev_honest"])[:15]:
    s = SYS[k]; print(f"{k:40s}{s['dev_honest']:8.4f}{s['dev_insample']:11.4f}  {s['members']}")

28 models: ['kd_student_m1x6_first1024', 'm1_adapter_first512', 'm1_full1024_hier', 'm1_full1024_hier_s1', 'm1_full1024_hier_s2', 'm1_full512_hier', 'm1_full_first1024', 'm1_full_first512', 'm1_ia3_first512', 'm1_lora_first1024', 'm1_lora_first512', 'm1_lora_headtail512', 'm1_lora_last512', 'm1_lora_middle512', 'm1_lorma_plus_first512', 'm1_normgain_first512', 'm1_partial1024_hier', 'm1_partial1024_hier_s1', 'm1_partial1024_hier_s2', 'm1_partial4_first1024', 'm1_partial4_first512', 'm1_prompt_first512', 'm1_rslora_first512', 'm1_unfreeze_first512', 'm2_lorma_plus_first512', 'm3_full_first512', 'm4_adapter_first512', 'nokd_student_m1x6_first1024']
ens_greedy                         honest global 0.4225 | perlabel 0.4242 | shrunk 0.4241  <- 7 members
ens_top5                           honest global 0.4227 | perlabel 0.4194 | shrunk 0.4253  <- 5 members
ens_hier                           honest global 0.4273 | perlabel 0.4263 | shrunk 0.4300  <- 7 members
kd_student_m1x6_first1024        

## P5-8 · Submission zips

Each system becomes p5_<name>.zip containing prediction.json = [{"id": ..., "pred": [statute ids]}]. Every case gets at least one statute. The ids are checked against the original test file: each test case appears exactly once, in the original order. UPLOAD_ORDER.txt ranks the zips: ensembles first, then single models, each best honest-dev first. Everything is also packed into one ALL_p5_zips.zip to download to Drive for the phone uploads from 22 Oct.

In [5]:
# ---- P5-8: submission zips (prediction.json = [{"id", "pred": [statute ids]}]) + validity checks ----
import json, os, zipfile, numpy as np, pickle
WORK = "/kaggle/working"
import p5_lsi as ls
SYS = json.load(open(f"{WORK}/stats/p5_systems.json"))
test_ids = [m["id"] for m in json.load(open(f"{WORK}/data_p5/clean/test.json"))]       # same order as the probs
raw = f"{WORK}/data_p5/Phase_5_LSI/Test/test.json"
if os.path.exists(raw):                                                      # cross-check against the original file
    assert [str(r["id"]) for r in json.load(open(raw))] == test_ids, "test id order mismatch!"
PT = {}
def ptest(m):
    if m not in PT:
        PT[m] = pickle.load(open(f"{WORK}/p5_prob/{m}.pkl", "rb"))["test"].astype(np.float32)
    return PT[m]

# ensembles (all rules) first, then single models, each group best honest dev score first
ens = sorted([k for k in SYS if k.startswith("ens_")], key=lambda k: -SYS[k]["dev_honest"])
single = sorted([k for k in SYS if not k.startswith("ens_")], key=lambda k: -SYS[k]["dev_honest"])
ORDER = ens + single
out_dir = f"{WORK}/submissions_p5"; os.makedirs(out_dir, exist_ok=True)
rows = []
for rank, k in enumerate(ORDER, 1):
    s = SYS[k]
    Pt = np.mean([ptest(m) for m in s["members"]], 0)
    D = ls.decide(Pt, np.array(s["T"]))
    pred = [{"id": i, "pred": [int(j) for j in np.flatnonzero(d)]} for i, d in zip(test_ids, D)]
    # validity: every test case exactly once, >= 1 label, ids in 0..99
    assert len(pred) == len(test_ids) == len({p["id"] for p in pred})
    assert all(p["pred"] and all(0 <= j < ls.NL for j in p["pred"]) for p in pred)
    folder = f"{out_dir}/p5_{k}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p5_{k}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    rows.append(f"{rank:2d}. p5_{k}.zip | dev honest {s['dev_honest']:.4f} | labels/case {D.sum(1).mean():.2f}")
open(f"{out_dir}/UPLOAD_ORDER.txt", "w").write("\n".join(rows) + "\n")
with zipfile.ZipFile(f"{out_dir}/ALL_p5_zips.zip", "w") as z:               # one file to download to Drive
    for k in ORDER:
        z.write(f"{out_dir}/p5_{k}/p5_{k}.zip", arcname=f"p5_{k}.zip")
    z.write(f"{out_dir}/UPLOAD_ORDER.txt", arcname="UPLOAD_ORDER.txt")
print("\n".join(rows)); print(f"\n{len(ORDER)} zips OK -> {out_dir}/ALL_p5_zips.zip")

 1. p5_ens_hier_shrunk.zip | dev honest 0.4300 | labels/case 4.36
 2. p5_ens_hier_global.zip | dev honest 0.4273 | labels/case 4.10
 3. p5_ens_hier_perlabel.zip | dev honest 0.4263 | labels/case 4.68
 4. p5_ens_top5_shrunk.zip | dev honest 0.4253 | labels/case 4.40
 5. p5_ens_greedy_perlabel.zip | dev honest 0.4242 | labels/case 4.73
 6. p5_ens_greedy_shrunk.zip | dev honest 0.4241 | labels/case 4.51
 7. p5_ens_top5_global.zip | dev honest 0.4227 | labels/case 4.18
 8. p5_ens_greedy_global.zip | dev honest 0.4225 | labels/case 4.36
 9. p5_ens_top5_perlabel.zip | dev honest 0.4194 | labels/case 4.67
10. p5_m1_full1024_hier_s2.zip | dev honest 0.4102 | labels/case 4.09
11. p5_m1_full1024_hier.zip | dev honest 0.4084 | labels/case 4.46
12. p5_m1_full1024_hier_s1.zip | dev honest 0.4079 | labels/case 4.08
13. p5_m1_partial1024_hier_s2.zip | dev honest 0.4029 | labels/case 4.05
14. p5_m1_partial1024_hier_s1.zip | dev honest 0.4015 | labels/case 3.96
15. p5_m1_partial1024_hier.zip | dev hone

## P5-9 · Analysis for the report

All of these use the best ensemble on the clean dev:

F1 for each statute vs how often it appears in training: are rare statutes the weak ones?
The most frequent co-occurring statute pairs: when only one of the pair is true, how often is the other wrongly added?
Predicted vs true labels per case, and F1 by number of true statutes and by judgment length.
Where the hierarchical models put their attention across the judgment.

In [6]:
# ---- P5-9: analysis for the report (on the best ensemble, CLEAN dev) ----
import glob, json, os, pickle, numpy as np
from collections import Counter
from scipy.stats import spearmanr
from sklearn.metrics import f1_score
WORK = "/kaggle/working"
import p5_lsi as ls
SYS = json.load(open(f"{WORK}/stats/p5_systems.json"))
DV = ls.MSplit(f"{WORK}/data_p5/tok/dev.npz", f"{WORK}/data_p5/clean/dev.json"); Y = DV.Y
best = max((k for k in SYS if k.startswith("ens_")), key=lambda k: SYS[k]["dev_honest"]); S = SYS[best]
Pd = np.mean([pickle.load(open(f"{WORK}/p5_prob/{m}.pkl", "rb"))["dev"].astype(np.float32) for m in S["members"]], 0)
D = ls.decide(Pd, np.array(S["T"])); A = {"system": best, "members": S["members"]}
print(f"best system: {best} | members {S['members']}")

# 1) per-statute F1 vs training frequency (are rare statutes the weak ones?)
tr = json.load(open(f"{WORK}/data_p5/clean/train.json"))
freq = np.bincount([l for r in tr for l in r["labels"]], minlength=ls.NL)
f_lab = f1_score(Y, D, average=None, zero_division=0)
rho = spearmanr(freq, f_lab).correlation
q = np.quantile(freq, [0.25, 0.5, 0.75])
bucket = np.digitize(freq, q)
A["f1_by_freq_quartile"] = [round(float(f_lab[bucket == b].mean()), 4) for b in range(4)]
A["spearman_freq_f1"] = round(float(rho), 3)
print(f"\nper-statute F1 by train-frequency quartile (rare -> common): {A['f1_by_freq_quartile']} | Spearman rho {rho:.3f}")
o = np.argsort(f_lab)
print("weakest 8 statutes:", [(int(j), int(freq[j]), round(float(f_lab[j]), 3)) for j in o[:8]], "(id, train cases, F1)")
print("strongest 8       :", [(int(j), int(freq[j]), round(float(f_lab[j]), 3)) for j in o[-8:]])
A["per_statute"] = {int(j): {"train_cases": int(freq[j]), "f1": round(float(f_lab[j]), 4)} for j in range(ls.NL)}

# 2) co-occurring statutes: does the model keep pairs apart?
pairs = Counter((a, b) for r in tr for i, a in enumerate(sorted(r["labels"])) for b in sorted(r["labels"])[i + 1:])
A["pairs"] = []
print("\npair (a,b) | dev cases with both | both predicted | only-a cases: b wrongly added | only-b cases: a wrongly added")
for (a, b), n in pairs.most_common(6):
    both = (Y[:, a] == 1) & (Y[:, b] == 1); oa = (Y[:, a] == 1) & (Y[:, b] == 0); ob = (Y[:, a] == 0) & (Y[:, b] == 1)
    r = dict(pair=[int(a), int(b)], n_both=int(both.sum()),
             both_pred=round(float((D[both, a] & D[both, b]).mean()), 3) if both.any() else None,
             b_added_when_only_a=round(float(D[oa, b].mean()), 3) if oa.any() else None,
             a_added_when_only_b=round(float(D[ob, a].mean()), 3) if ob.any() else None)
    A["pairs"].append(r); print(r)

# 3) cardinality + difficulty by #true labels and by judgment length (per-case F1)
inter = (D * Y).sum(1); case_f1 = 2 * inter / np.maximum(D.sum(1) + Y.sum(1), 1)
nt = Y.sum(1); L = np.diff(DV.off)
A["labels_per_case"] = {"true": round(float(nt.mean()), 2), "pred": round(float(D.sum(1).mean()), 2)}
A["case_f1_by_true_count"] = {k: round(float(case_f1[m].mean()), 4) for k, m in
                             {"1": nt == 1, "2-3": (nt >= 2) & (nt <= 3), "4-6": (nt >= 4) & (nt <= 6), "7+": nt >= 7}.items() if m.any()}
A["case_f1_by_length"] = {k: round(float(case_f1[m].mean()), 4) for k, m in
                         {"<=512": L <= 512, "513-1024": (L > 512) & (L <= 1024), "1025-4096": (L > 1024) & (L <= 4096),
                          ">4096": L > 4096}.items() if m.any()}
print("\nlabels/case:", A["labels_per_case"], "\ncase F1 by #true statutes:", A["case_f1_by_true_count"],
      "\ncase F1 by judgment length (tokens):", A["case_f1_by_length"])

# 4) where the hierarchical models look (attention mass per position decile, cases with >= 3 chunks)
A["attention"] = {}
for f in sorted(glob.glob(f"{WORK}/stats/p5_runs/*_hier*.json")):
    R = json.load(open(f)); A["attention"][os.path.basename(f)[:-5]] = R.get("attention_by_position_decile")
for k, v in A["attention"].items():
    print(f"attention {k:28s} {v}")

json.dump(A, open(f"{WORK}/stats/p5_analysis.json", "w"), indent=1); print("\nsaved stats/p5_analysis.json")

best system: ens_hier_shrunk | members ['m1_full1024_hier', 'm1_full1024_hier_s1', 'm1_full1024_hier_s2', 'm1_full512_hier', 'm1_partial1024_hier', 'm1_partial1024_hier_s1', 'm1_partial1024_hier_s2']

per-statute F1 by train-frequency quartile (rare -> common): [0.3201, 0.4162, 0.4387, 0.561] | Spearman rho 0.557
weakest 8 statutes: [(8, 346, 0.106), (0, 341, 0.128), (32, 382, 0.168), (2, 367, 0.17), (52, 1068, 0.191), (99, 655, 0.192), (9, 812, 0.195), (40, 531, 0.204)] (id, train cases, F1)
strongest 8       : [(61, 3673, 0.671), (37, 1251, 0.674), (29, 495, 0.703), (81, 1255, 0.737), (94, 3786, 0.754), (20, 467, 0.778), (28, 582, 0.823), (34, 10259, 0.849)]

pair (a,b) | dev cases with both | both predicted | only-a cases: b wrongly added | only-b cases: a wrongly added
{'pair': [5, 34], 'n_both': 907, 'both_pred': 0.66, 'b_added_when_only_a': 0.08, 'a_added_when_only_b': 0.213}
{'pair': [12, 13], 'n_both': 846, 'both_pred': 0.606, 'b_added_when_only_a': 0.319, 'a_added_when_only_b'

## P5-B3 · Final Phase 5 backup

Uploads the submissions, systems and analysis to HF. Then download submissions_p5/ALL_p5_zips.zip from Kaggle’s Output panel (right side → ⋮ → Download) and put it on Google Drive for the phone uploads.

In [7]:
# ---- P5-B3: final backup of zips + systems + analysis (token from the Kaggle secret only) ----
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
WORK = "/kaggle/working"
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 5 final: zips + systems + analysis", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["submissions_p5/**", "stats/**", "p5_prob/**", "src/**"])
print("Phase 5 backed up ✅ -> now download submissions_p5/ALL_p5_zips.zip to Google Drive")

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Phase 5 backed up ✅ -> now download submissions_p5/ALL_p5_zips.zip to Google Drive


## PX · Phase 4 + 5 final upgrades
Phase 5, train + clean dev: the final full and partial fine-tunes (first-1024, 2 epochs) are retrained with the 10k clean-dev cases added (+24% data). Their hierarchical models (3 pooler seeds each) are rebuilt on top for each one’s best epoch. These can’t be scored on dev, so their thresholds come from the dev-validated twins. A rate-matched variant is a hedge against a calibration shift: it shifts the thresholds so the model predicts the same number of labels per case as its twin, using test predictions only, never labels.
Phase 5, label-correlation stacker: for every statute, a logistic regression sees the ensemble’s scores for all 100 statutes, so it can learn co-occurrence patterns such as (12, 13). It’s scored honestly with 5-fold out-of-fold predictions on dev, and a zip is written only if it really helps.
Phase 4, pooler seeds: 2 extra seeds for each of the 3 hierarchical models, using the cached chunk embeddings, then new ensembles. Only clean dev is used (dev without the test cases).

In [9]:
# ---- PX-0: make sure Phase-4 + Phase-5 files are here (restore only what is missing from HF) ----
import os, shutil, torch
WORK, REPO = "/kaggle/working", "Utkarsh-1718-1/cs781-phase1-work"
need = {"src/p5_lsi.py": "src/**", "src/p4_run.py": "src/**", "checkpoints/m1_dense_100m/best.pt": "checkpoints/m1_dense_100m/best.pt",
        "data_p5/tok/train.npz": "data_p5/tok/**", "data_p5/clean/train.json": "data_p5/clean/**",
        "stats/p5_runs/m1_full_first1024.json": "stats/**", "stats/p5_systems.json": "stats/**",
        "p5_prob/m1_full1024_hier.pkl": "p5_prob/**", "submissions_p5/UPLOAD_ORDER.txt": "submissions_p5/**",
        "data_p4/tok/dev.npz": "data_p4/tok/**", "data_p4/clean/test.json": "data_p4/clean/**",
        "p4_prob/m1_full1024_hier.pkl": "p4_prob/**", "stats/p4_runs/m1_full1024_hier.json": "stats/**",
        "p4_chunks/m1_full_last512.pkl": "p4_chunks/m1_full_last512.pkl",
        "p4_chunks/m1_full_last1024.pkl": "p4_chunks/m1_full_last1024.pkl",
        "p4_chunks/m1_partial4_last512.pkl": "p4_chunks/m1_partial4_last512.pkl"}
missing = sorted({pat for p, pat in need.items() if not os.path.exists(f"{WORK}/{p}")})
print("to download:", missing or "nothing")
if missing:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import snapshot_download
    snapshot_download(REPO, repo_type="model", local_dir=WORK, allow_patterns=missing,
                      token=UserSecretsClient().get_secret("HF_WRITE_TOKEN"))
    print("still missing:", [p for p in need if not os.path.exists(f"{WORK}/{p}")] or "nothing ✅")
print(f"GPUs: {torch.cuda.device_count()} | free disk: {shutil.disk_usage(WORK).free/1e9:.1f} GB (need ~3 GB)")

to download: nothing
GPUs: 2 | free disk: 2.4 GB (need ~3 GB)


In [10]:
%%writefile /kaggle/working/src/p5_td.py
"""
p5_td.py -- Phase-5 final models retrained on TRAIN + CLEAN DEV (background, one process per GPU):

    CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup python p5_td.py --plan plan.json >> log 2>&1 &
    python p5_td.py --smoke        # ~2 min check of every code path on a few cases (run first)

Jobs:
  {"type": "td", "name": X}        X's exact config (stats/p5_runs/X.json) trained on train + clean dev for
                                   X's full epoch count (no held-out data -> last step kept)  -> X_td
  {"type": "hier_td", "name": H}   hierarchical model H rebuilt on the chunk embeddings of <H's base>_td,
                                   trained on train + dev for H's best epoch, same seed      -> H_td
Outputs: p5_ckpt/<n>_td.pt, p5_prob/<n>_td.pkl ({"test"} only: these models have seen dev),
stats/p5_runs/<n>_td.json (written last = done marker). Test labels do not exist on disk.
"""
import argparse
import json
import math
import os
import pickle
import random
import re
import sys
import time
import traceback

import numpy as np
import torch
import torch.nn.functional as F

WORK = "/kaggle/working"
sys.path.insert(0, f"{WORK}/src")
import p5_lsi as ls                                                   # noqa: E402
import p4_cjpe as cj                                                  # noqa: E402
import p2_peft as pp                                                  # noqa: E402

ap = argparse.ArgumentParser(); ap.add_argument("--plan"); ap.add_argument("--smoke", action="store_true")
ARGS = ap.parse_args()
DEV = "cuda:0"
CK = {"m1": f"{WORK}/checkpoints/m1_dense_100m/best.pt", "m2": f"{WORK}/checkpoints/m2_mla_moe/best.pt",
      "m3": f"{WORK}/checkpoints/m3_encdec/best.pt", "m4": f"{WORK}/checkpoints/m4_copy/best.pt"}
OUT, PRB, CKD, CHK = (f"{WORK}/stats/p5_runs", f"{WORK}/p5_prob", f"{WORK}/p5_ckpt", f"{WORK}/p5_chunks")
for d in (OUT, PRB, CKD, CHK):
    os.makedirs(d, exist_ok=True)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


class Cat:
    """Several splits glued into one (tokens, offsets, multi-hot labels)."""
    def __init__(self, *parts):
        self.ids = np.concatenate([p.ids[:p.off[p.n]] for p in parts])
        offs, base = [np.zeros(1, dtype=np.int64)], 0
        for p in parts:
            offs.append(base + p.off[1:p.n + 1].astype(np.int64)); base += int(p.off[p.n])
        self.off = np.concatenate(offs)
        self.Y = np.vstack([p.Y[:p.n] for p in parts]) if all(p.Y is not None for p in parts) else None
        self.labels, self.n = self.Y, sum(p.n for p in parts)

    def arr(self, i):
        return self.ids[self.off[i]:self.off[i + 1]]


class Head:
    """First n cases of a split (smoke test only)."""
    def __init__(self, s, n):
        self.ids, self.off, self.n = s.ids, s.off, n
        self.Y = None if s.Y is None else s.Y[:n]

    def arr(self, i):
        return self.ids[self.off[i]:self.off[i + 1]]


TR, DV, TE = (ls.MSplit(f"{WORK}/data_p5/tok/{s}.npz", f"{WORK}/data_p5/clean/{s}.json")
              for s in ("train", "dev", "test"))
TD = Cat(TR, DV)
log(f"data: train {TR.n} + clean dev {DV.n} = {TD.n} cases | test {TE.n}")


def make_model(ckpt, method, student_layers=None, partial_k=4, **kw):
    m = ls.build(CK[ckpt], method, bidirectional=True, partial_k=partial_k, **kw)
    if student_layers:
        pp.shrink_dense(m.tagger, tuple(student_layers))
    return m


def split_cfg(cfg):
    """run config -> (model kwargs, training kwargs)"""
    c = {k: v for k, v in cfg.items() if v is not None}
    tr = {k: c.pop(k) for k in ("mode", "k", "epochs", "lr", "head_lr", "bs", "seed", "layerwise_unfreeze") if k in c}
    for k in ("teachers", "kd_T", "kd_lam"):
        c.pop(k, None)
    return c, tr


def train_td(model, tr, mode="first", k=1024, epochs=2, lr=3e-5, head_lr=1e-3, bs=8, seed=0, warmup=0.03,
             layerwise_unfreeze=False, max_steps=None):
    """Same optimiser / loss / schedule as p5_lsi.train, but no evaluation: the last step is kept."""
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    model.to(DEV)
    pos = tr.Y.sum(0); pw = torch.tensor(np.sqrt((tr.n - pos) / np.maximum(pos, 1)), dtype=torch.float, device=DEV)
    layers = list(pp._layers(model.tagger))
    if layerwise_unfreeze:
        for p in model.tagger.backbone.parameters():
            p.requires_grad = False
    head_ids = {id(p) for p in model.head().parameters()}
    opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in head_ids
                                         and (p.requires_grad or layerwise_unfreeze)], "lr": lr},
                             {"params": list(model.head().parameters()), "lr": head_lr}], weight_decay=0.01)
    steps = epochs * math.ceil(tr.n / bs); warm = max(1, int(warmup * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")
    step, t0, run = 0, time.time(), 0.0
    log(f"trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.3f}M | {tr.n} cases | "
        f"{steps} steps | view {mode}-{k} | no dev (train+dev)")
    for ep in range(epochs):
        order = np.random.permutation(tr.n)
        for b in range(0, tr.n, bs):
            if layerwise_unfreeze:
                n_on = min(len(layers), 1 + int(step / max(1, steps) * len(layers)))
                for L in layers[len(layers) - n_on:]:
                    for p in L.parameters():
                        p.requires_grad = True
            model.train(); bi = order[b:b + bs]
            ids, mask = cj.collate([cj.section(tr.arr(i), mode, k) for i in bi], DEV)
            with cj._autocast(DEV):
                z = model(ids, mask).float()
            loss = F.binary_cross_entropy_with_logits(z, torch.from_numpy(tr.Y[bi]).to(DEV), pos_weight=pw)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step(); step += 1; run += loss.item()
            if step % 500 == 0:
                log(f"   step {step}/{steps} | loss {run/500:.4f} | {time.time()-t0:.0f}s"); run = 0.0
            if max_steps and step >= max_steps:
                return
    return


def td_job(base, smoke=False):
    t0, name = time.time(), f"{base}_td"
    R = json.load(open(f"{OUT}/{base}.json")); mk, tk = split_cfg(R["cfg"])
    method = mk["method"]
    model = make_model(mk.pop("ckpt"), mk.pop("method"), mk.pop("student_layers", None), mk.pop("partial_k", 4), **mk)
    log(f"=== {name}: {base}'s config on train + clean dev | {tk}")
    tr, te = (Cat(Head(TR, 12), Head(DV, 4)), Head(TE, 6)) if smoke else (TD, TE)
    train_td(model, tr, **tk, max_steps=3 if smoke else None)
    pt = ls.predict(model, te, tk.get("mode", "first"), tk.get("k", 1024), DEV)
    if smoke:
        log("td ok: test probs", pt.shape); return model
    torch.save({n: p.detach().half().cpu() for n, p in model.named_parameters()
                if p.requires_grad or tk.get("layerwise_unfreeze") or method in ("full", "partial")}, f"{CKD}/{name}.pt")
    pickle.dump({"test": pt.astype(np.float16)}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump({"cfg": R["cfg"], "base": base, "td": True, "min": round((time.time() - t0) / 60, 1),
               "test_labels_per_case@0.5": float(ls.decide(pt, 0.5).sum(1).mean())},
              open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name} | {round((time.time() - t0) / 60, 1)} min")
    del model; torch.cuda.empty_cache()


def _load_td_base(base_td):
    R = json.load(open(f"{OUT}/{base_td[:-3]}.json")); mk, tk = split_cfg(R["cfg"])
    m = make_model(mk.pop("ckpt"), mk.pop("method"), mk.pop("student_layers", None), mk.pop("partial_k", 4), **mk)
    sd = {n: v.float() for n, v in torch.load(f"{CKD}/{base_td}.pt", map_location="cpu").items()}
    m.load_state_dict(sd, strict=False)
    return m.to(DEV)


def _pad(E, idx):
    C = max(len(E[i]) for i in idx)
    x = np.zeros((len(idx), C, E[idx[0]].shape[1]), dtype=np.float32); mk = np.zeros((len(idx), C), dtype=bool)
    for b, i in enumerate(idx):
        x[b, :len(E[i])] = E[i]; mk[b, :len(E[i])] = True
    return torch.from_numpy(x).to(DEV), torch.from_numpy(mk).to(DEV)


@torch.no_grad()
def _pool_predict(pool, E, bs=64):
    pool.eval(); p = np.zeros((len(E), ls.NL), dtype=np.float32)
    order = np.argsort([len(e) for e in E])
    for b in range(0, len(E), bs):
        idx = order[b:b + bs]; x, mk = _pad(E, idx)
        p[idx] = torch.sigmoid(pool(x, mk)).cpu().numpy()
    return p


def hier_td_job(hname, smoke_model=None):
    t0, name = time.time(), f"{hname}_td"
    if smoke_model is not None:                                       # smoke: 6 train+dev cases, 3 test cases
        tr, te, m = Cat(Head(TR, 4), Head(DV, 2)), Head(TE, 3), smoke_model
        E = {"td": cj.encode_chunks(m, tr, 1024, DEV, max_chunks=24), "test": cj.encode_chunks(m, te, 1024, DEV, max_chunks=24)}
        cfg, best_ep, seed, Y = {"lr": 1e-3}, 1, 0, tr.Y
    else:
        R = json.load(open(f"{OUT}/{hname}.json")); cfg = R["cfg"]
        best_ep = int(max(R["hist"], key=lambda h: h["macro_f1"])["epoch"])
        seed = int(re.search(r"_s(\d+)$", hname).group(1)) if re.search(r"_s(\d+)$", hname) else 0
        base_td = cfg["base"] + "_td"; cache = f"{CHK}/{base_td}.pkl"
        if os.path.exists(cache):
            E = pickle.load(open(cache, "rb")); log(f"   loaded chunk embeddings of {base_td}")
        else:
            m = _load_td_base(base_td); E = {}
            for s, sp in (("test", TE), ("td", TD)):
                ti = time.time(); E[s] = cj.encode_chunks(m, sp, cfg["chunk"], DEV, max_chunks=cfg["max_chunks"])
                log(f"   encoded {s}: {sum(len(e) for e in E[s])} chunks in {time.time()-ti:.0f}s")
            del m; torch.cuda.empty_cache(); pickle.dump(E, open(cache, "wb"))
        Y = TD.Y
    torch.manual_seed(seed); np.random.seed(seed)
    n = len(E["td"]); pool = ls.ChunkPoolerML(E["td"][0].shape[1]).to(DEV)
    pos = Y.sum(0); pw = torch.tensor(np.sqrt((n - pos) / np.maximum(pos, 1)), dtype=torch.float, device=DEV)
    bs = 32; steps = best_ep * int(np.ceil(n / bs))
    opt = torch.optim.AdamW(pool.parameters(), lr=cfg["lr"], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg["lr"], total_steps=steps, pct_start=0.1)
    log(f"=== {name}: pooler on train+dev chunk embeddings | {best_ep} epochs | seed {seed}")
    for ep in range(best_ep):
        pool.train(); order = np.random.permutation(n)
        for b in range(0, n, bs):
            idx = order[b:b + bs]; x, mk = _pad(E["td"], idx)
            loss = F.binary_cross_entropy_with_logits(pool(x, mk), torch.from_numpy(Y[idx]).to(DEV), pos_weight=pw)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(pool.parameters(), 1.0); opt.step(); sched.step()
    pt = _pool_predict(pool, E["test"])
    if smoke_model is not None:
        log("hier_td ok: test probs", pt.shape); return
    torch.save(pool.state_dict(), f"{CKD}/{name}.pt")
    pickle.dump({"test": pt.astype(np.float16)}, open(f"{PRB}/{name}.pkl", "wb"))
    json.dump({"cfg": {**cfg, "type": "hier_td", "ref": hname}, "td": True, "epochs_trained": best_ep, "seed": seed,
               "min": round((time.time() - t0) / 60, 1)}, open(f"{OUT}/{name}.json", "w"), indent=1)
    log(f"DONE {name} | {round((time.time() - t0) / 60, 1)} min")


if ARGS.smoke:
    m = td_job("m1_partial4_first1024", smoke=True)
    hier_td_job("smoke", smoke_model=m)
    log("SMOKE TEST PASSED"); sys.exit(0)

for job in json.load(open(ARGS.plan)):
    name, typ = job["name"], job["type"]
    if os.path.exists(f"{OUT}/{name}_td.json"):
        log(f"skip {name}_td (already done)"); continue
    try:
        td_job(name) if typ == "td" else hier_td_job(name)
    except Exception:
        log(f"!!! {name}_td FAILED"); traceback.print_exc(); torch.cuda.empty_cache()
log("PLAN DONE")

Writing /kaggle/working/src/p5_td.py


In [11]:
# ---- PX-1b: smoke test (~2 min, a few cases): must end with SMOKE TEST PASSED ----
!cd /kaggle/working/src && python p5_td.py --smoke 2>&1 | tail -n 12

09:22:55 data: train 42102 + clean dev 10085 = 52187 cases | test 13019
09:22:57 === m1_partial4_first1024_td: m1_partial4_first1024's config on train + clean dev | {'mode': 'first', 'k': 1024, 'epochs': 2, 'lr': 0.0001, 'head_lr': 0.001, 'bs': 8, 'seed': 0, 'layerwise_unfreeze': False}
09:22:58 trainable 25.249M | 16 cases | 4 steps | view first-1024 | no dev (train+dev)
09:23:00 td ok: test probs (6, 100)
09:23:00 === smoke_td: pooler on train+dev chunk embeddings | 1 epochs | seed 0
09:23:00 hier_td ok: test probs (3, 100)
09:23:00 SMOKE TEST PASSED


In [12]:
# ---- PX-2: launch. GPU 0: Phase-5 full model on train+dev. GPU 1: Phase-4 seeds, then Phase-5 partial on train+dev ----
import json, os, subprocess
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
assert __import__("torch").cuda.device_count() == 2, "turn on the T4 x2 accelerator first"
H4 = dict(type="hier", epochs=8)
p4 = [dict(name=f"m1_full512_hier_s{s}", base="m1_full_last512", seed=s, **H4) for s in (1, 2)] + \
     [dict(name=f"m1_full1024_hier_s{s}", base="m1_full_last1024", chunk=1024, max_chunks=32, seed=s, **H4) for s in (1, 2)] + \
     [dict(name=f"m1_partial512_hier_s{s}", base="m1_partial4_last512", seed=s, **H4) for s in (1, 2)]
p5 = {0: [dict(type="td", name="m1_full_first1024")] +
         [dict(type="hier_td", name=n) for n in ("m1_full1024_hier", "m1_full1024_hier_s1", "m1_full1024_hier_s2")],
      1: [dict(type="td", name="m1_partial4_first1024")] +
         [dict(type="hier_td", name=n) for n in ("m1_partial1024_hier", "m1_partial1024_hier_s1", "m1_partial1024_hier_s2")]}
json.dump(p4, open(f"{WORK}/stats/p4_seed_plan.json", "w"), indent=1)
for g in (0, 1):
    json.dump(p5[g], open(f"{WORK}/stats/p5_td_gpu{g}.json", "w"), indent=1)
if sh("pgrep -f '[p]5_td.py --plan|[p]4_run.py --plan'"):
    print("already running - not relaunched")
else:
    run5 = lambda g: f"python p5_td.py --plan {WORK}/stats/p5_td_gpu{g}.json >> {WORK}/stats/p5_td_gpu{g}.log 2>&1"
    run4 = f"python p4_run.py --plan {WORK}/stats/p4_seed_plan.json >> {WORK}/stats/p4_seed.log 2>&1"
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES=0 PYTHONUNBUFFERED=1 setsid nohup {run5(0)} < /dev/null &", shell=True)
    subprocess.run(f"cd {WORK}/src && CUDA_VISIBLE_DEVICES=1 PYTHONUNBUFFERED=1 setsid nohup bash -c "
                   f"'{run4}; {run5(1)}' < /dev/null > /dev/null 2>&1 &", shell=True)        # one after the other
    print("GPU 0: Phase-5 full train+dev + 3 poolers | GPU 1: 6 Phase-4 pooler seeds -> Phase-5 partial train+dev + 3 poolers")

GPU 0: Phase-5 full train+dev + 3 poolers | GPU 1: 6 Phase-4 pooler seeds -> Phase-5 partial train+dev + 3 poolers


In [13]:
# ---- P5-LC: label-correlation stacker on the best ensemble (CPU, honest cross-fitting on dev) ----
import sys, json, os, pickle, zipfile, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p5_lsi as ls
Y = ls.MSplit(f"{WORK}/data_p5/tok/dev.npz", f"{WORK}/data_p5/clean/dev.json").Y
SYS = json.load(open(f"{WORK}/stats/p5_systems.json")); REF = "ens_hier_shrunk"
mem = SYS[REF]["members"]
load = lambda m, s: pickle.load(open(f"{WORK}/p5_prob/{m}.pkl", "rb"))[s].astype(np.float32)
Pd = np.mean([load(m, "dev") for m in mem], 0); Pt = np.mean([load(m, "test") for m in mem], 0)
logit = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))

# same metric / threshold rules / honest halves as P5-7
TS = np.linspace(0.05, 0.9, 35)
def mf1(Yt, D):
    D = D.astype(np.float32); tp = (D * Yt).sum(0); fp = (D * (1 - Yt)).sum(0); fn = ((1 - D) * Yt).sum(0)
    return float(np.mean(2 * tp / np.maximum(2 * tp + fp + fn, 1)))
def tune_t(Yt, P):
    f = [mf1(Yt, ls.decide(P, t)) for t in TS]; i = int(np.argmax(f)); return f[i], float(TS[i])
def thresholds(Yt, P, rule):
    t = tune_t(Yt, P)[1]
    if rule == "global":
        return np.full(ls.NL, t)
    T = ls.tune_per_label(Yt, P)
    return T if rule == "perlabel" else 0.5 * (T + t)
perm = np.random.RandomState(0).permutation(len(Y)); h = len(Y) // 2
HALVES = [(perm[:h], perm[h:]), (perm[h:], perm[:h])]
def honest(P):
    return {r: float(np.mean([mf1(Y[b], ls.decide(P[b], thresholds(Y[a], P[a], r))) for a, b in HALVES]))
            for r in ("global", "perlabel", "shrunk")}

# stacker: for every statute j, logistic regression on the logits of ALL 100 statutes
def fit_predict(Xa, Ya, Xb):
    out = np.zeros((len(Xb), ls.NL), np.float32)
    for j in range(ls.NL):
        clf = LogisticRegression(C=0.3, max_iter=500); clf.fit(Xa, Ya[:, j]); out[:, j] = clf.predict_proba(Xb)[:, 1]
    return out
Xd, Xt = logit(Pd), logit(Pt)
OOF = np.zeros_like(Pd)                                               # every dev case predicted by a stacker
for k, (a, b) in enumerate(KFold(5, shuffle=True, random_state=1).split(Xd)):   # that never saw it
    OOF[b] = fit_predict(Xd[a], Y[a], Xd[b]); print(f"   fold {k+1}/5 done")

CAND = {"base (ens_hier)": Pd, "stacked": OOF, "blend 50/50": 0.5 * (Pd + OOF)}
H = {n: honest(P) for n, P in CAND.items()}
for n, s in H.items():
    print(f"{n:18s} honest dev macro-F1: " + " | ".join(f"{r} {v:.4f}" for r, v in s.items()))
base_best = max(H["base (ens_hier)"].values())
best_n, best_r = max(((n, r) for n in ("stacked", "blend 50/50") for r in H[n]), key=lambda x: H[x[0]][x[1]])
gain = H[best_n][best_r] - base_best
print(f"\nbest: {best_n} / {best_r}: {H[best_n][best_r]:.4f} ({gain:+.4f} vs {base_best:.4f})")

if gain > 0.002:                                                       # only worth a submission if it helps
    St = fit_predict(Xd, Y, Xt)                                        # final stacker: all of dev -> test
    Pt_new = St if best_n == "stacked" else 0.5 * (Pt + St)
    Pd_new = CAND[best_n]
    T = thresholds(Y, Pd_new, best_r)
    D = ls.decide(Pt_new, T)
    test_ids = [m["id"] for m in json.load(open(f"{WORK}/data_p5/clean/test.json"))]
    pred = [{"id": i, "pred": [int(j) for j in np.flatnonzero(d)]} for i, d in zip(test_ids, D)]
    assert len({p["id"] for p in pred}) == len(test_ids) and all(p["pred"] for p in pred)
    name = f"ens_hier_lc_{best_r}"; folder = f"{WORK}/submissions_p5/p5_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p5_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    json.dump({"name": name, "variant": best_n, "rule": best_r, "dev_honest": H[best_n][best_r], "base": base_best},
              open(f"{WORK}/stats/p5_lc.json", "w"), indent=1)
    print(f"wrote p5_{name}.zip | test labels/case {D.sum(1).mean():.2f}")
else:
    json.dump({"name": None, "honest": H, "base": base_best}, open(f"{WORK}/stats/p5_lc.json", "w"), indent=1)
    print("no real gain -> no new zip (the analysis still goes in the report: co-occurrence is not linearly fixable)")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

   fold 1/5 done


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

   fold 2/5 done


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

   fold 3/5 done


... (2 outputs left out)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

   fold 5/5 done
base (ens_hier)    honest dev macro-F1: global 0.4273 | perlabel 0.4263 | shrunk 0.4300
stacked            honest dev macro-F1: global 0.4039 | perlabel 0.4088 | shrunk 0.4101
blend 50/50        honest dev macro-F1: global 0.4315 | perlabel 0.4301 | shrunk 0.4341

best: blend 50/50 / shrunk: 0.4341 (+0.0041 vs 0.4300)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

wrote p5_ens_hier_lc_shrunk.zip | test labels/case 4.40


In [14]:
# ---- P4-S2: wait for the Phase-4 pooler seeds -> new ensembles (honest 2-fold on clean dev) -> zips ----
import sys, os, re, json, glob, pickle, zipfile, time, subprocess, numpy as np
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p4_cjpe as cj
LOG = f"{WORK}/stats/p4_seed.log"                                       # the seeds run first on GPU 1
while not (os.path.exists(LOG) and "PLAN DONE" in open(LOG).read()):
    print(time.strftime("%H:%M"), "UTC: Phase-4 seeds still training ..."); time.sleep(60)

y = cj.Split(f"{WORK}/data_p4/tok/dev.npz").labels                     # CLEAN dev (test cases removed)
P = {}
for f in glob.glob(f"{WORK}/p4_prob/*.pkl"):
    d = pickle.load(open(f, "rb"))
    if "dev" in d and "test" in d:
        P[os.path.basename(f)[:-4]] = {s: np.asarray(d[s], np.float32) for s in ("dev", "test")}
print(len(P), "Phase-4 models:", sorted(P))
def score(yy, p):
    t = cj.tune_threshold(yy, p); return cj.metrics(yy, p, t)["macro_f1"], t
def greedy(idx, names, steps=6):
    sel, best, s = [], -1.0, None
    for _ in range(steps):
        f, n = max((score(y[idx], P[m]["dev"][idx] if s is None else (s + P[m]["dev"][idx]) / (len(sel) + 1))[0], m)
                   for m in names)
        if f <= best + 1e-4:
            break
        sel.append(n); best = f; s = P[n]["dev"][idx] if s is None else s + P[n]["dev"][idx]
    return sel
perm = np.random.RandomState(0).permutation(len(y)); h = len(y) // 2
HALVES = [(perm[:h], perm[h:]), (perm[h:], perm[:h])]
def honest(fn):                                       # choose members + threshold on one half, score the other
    out = []
    for a, b in HALVES:
        mem = fn(a); pa = np.mean([P[m]["dev"][a] for m in mem], 0); pb = np.mean([P[m]["dev"][b] for m in mem], 0)
        out.append(cj.metrics(y[b], pb, cj.tune_threshold(y[a], pa))["macro_f1"])
    return float(np.mean(out))

seeds = [n for n in P if re.search(r"_s\d+$", n)]
no_seed = [n for n in P if n not in seeds]
hier = [n for n in P if "_hier" in n]
SYSTEMS = {"ens_greedy_noseeds (= old method)": lambda idx: greedy(idx, no_seed),
           "ens_greedy_seeds": lambda idx: greedy(idx, list(P)),
           "ens_hier_all": lambda idx: hier}
test_ids = [r["id"] for r in json.load(open(f"{WORK}/data_p4/clean/test.json"))]
rows = []
for name, fn in SYSTEMS.items():
    mem = fn(np.arange(len(y))); pd_ = np.mean([P[m]["dev"] for m in mem], 0); f, t = score(y, pd_)
    rows.append((name, honest(fn), f, t, mem))
    if "old method" in name:
        continue                                                     # already submitted as ens_greedy
    pt = np.mean([P[m]["test"] for m in mem], 0)
    pred = [{"id": i, "pred": int(v >= t)} for i, v in zip(test_ids, pt)]
    assert len(pred) == len({p["id"] for p in pred})
    folder = f"{WORK}/submissions_p4/p4_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p4_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
print(f"\n{'system':36s}{'honest dev':>11s}{'in-sample':>11s}{'t':>6s}  members")
for name, hd, f, t, mem in rows:
    print(f"{name:36s}{hd:11.4f}{f:11.4f}{t:6.2f}  {mem}")
json.dump([{"system": r[0], "honest": r[1], "insample": r[2], "t": r[3], "members": r[4]} for r in rows],
          open(f"{WORK}/stats/p4_seed_ensembles.json", "w"), indent=1)
print("\nupload a new one only if its HONEST dev beats the 'old method' row (differences < 0.005 are noise)")

26 Phase-4 models: ['kd_student_m1x6_last512', 'm1_adapter_last512', 'm1_full1024_hier', 'm1_full1024_hier_s1', 'm1_full1024_hier_s2', 'm1_full512_hier', 'm1_full512_hier_s1', 'm1_full512_hier_s2', 'm1_full_last1024', 'm1_full_last512', 'm1_lora_first512', 'm1_lora_headtail512', 'm1_lora_last1024', 'm1_lora_last512', 'm1_lora_middle512', 'm1_lorma_plus_last512', 'm1_normgain_last512', 'm1_partial4_last512', 'm1_partial512_hier', 'm1_partial512_hier_s1', 'm1_partial512_hier_s2', 'm1_unfreeze_last512', 'm2_lorma_plus_last512', 'm3_full_last512', 'm4_adapter_last512', 'nokd_student_m1x6_last512']

system                               honest dev  in-sample     t  members
ens_greedy_noseeds (= old method)        0.8065     0.8189  0.42  ['m1_full1024_hier', 'm1_partial4_last512', 'm1_full512_hier']
ens_greedy_seeds                         0.8146     0.8228  0.37  ['m1_full1024_hier', 'm1_full512_hier_s1', 'm4_adapter_last512', 'm1_partial4_last512']
ens_hier_all                             

In [17]:
# ---- PX-D: is the GPU-0 job alive? what did it last do? why did it stop? ----
import subprocess, shutil
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print("processes:", sh("pgrep -af '[p]5_td.py|[p]4_run.py'") or "NONE running")
print(sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader"))
print(f"free disk: {shutil.disk_usage(WORK).free/1e9:.2f} GB | RAM: {sh('free -g | head -2')}")
print("\n--- last 15 lines of the GPU-0 log ---\n" + sh(f"tail -n 15 {WORK}/stats/p5_td_gpu0.log"))
print("\n--- kernel messages (out-of-memory kills) ---\n" + (sh("dmesg 2>/dev/null | grep -i -E 'killed process|out of memory' | tail -3") or "none visible"))
print("\nfinished td results:", sh(f"ls {WORK}/stats/p5_runs/ | grep _td"))

processes: NONE running
0, 0 %, 0 MiB
1, 0 %, 0 MiB
free disk: 1.41 GB | RAM: total        used        free      shared  buff/cache   available
Mem:              31           1           2           0          26          28

--- last 15 lines of the GPU-0 log ---
11:25:48    step 12000/13048 | loss 0.2081 | 7358s
11:30:54    step 12500/13048 | loss 0.2027 | 7664s
11:35:59    step 13000/13048 | loss 0.2051 | 7969s
11:39:42 DONE m1_full_first1024_td | 136.6 min
11:50:53    encoded test: 30121 chunks in 668s
12:34:10    encoded td: 117284 chunks in 2597s
12:34:10 === m1_full1024_hier_td: pooler on train+dev chunk embeddings | 9 epochs | seed 0
12:35:20 DONE m1_full1024_hier_td | 55.6 min
12:35:20    loaded chunk embeddings of m1_full_first1024_td
12:35:20 === m1_full1024_hier_s1_td: pooler on train+dev chunk embeddings | 10 epochs | seed 1
12:36:37 DONE m1_full1024_hier_s1_td | 1.3 min
12:36:37    loaded chunk embeddings of m1_full_first1024_td
12:36:37 === m1_full1024_hier_s2_td: pooler

In [18]:
# ---- PX-W: keep the session busy until all GPU work is done, then back up everything ----
import subprocess, time
from IPython.display import clear_output
WORK = "/kaggle/working"
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
while sh("pgrep -f '[p]5_td.py --plan|[p]4_run.py --plan'"):
    clear_output(wait=True)
    print(f"[{time.strftime('%H:%M', time.gmtime(time.time() + 19800))} IST]")
    for f in ("p5_td_gpu0", "p4_seed", "p5_td_gpu1"):
        print(f"{f:11s}: {sh(f'grep -c DONE {WORK}/stats/{f}.log 2>/dev/null') or 0} done | "
              f"{sh(f'grep -c FAILED {WORK}/stats/{f}.log 2>/dev/null') or 0} failed | {sh(f'tail -n 1 {WORK}/stats/{f}.log 2>/dev/null')[:120]}")
    time.sleep(120)
print(sh(f"grep -h -E 'DONE|FAILED' {WORK}/stats/p5_td_gpu0.log {WORK}/stats/p4_seed.log {WORK}/stats/p5_td_gpu1.log"))
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 4 seeds + phase 5 train+dev", ignore_patterns=["**/__pycache__/**"],
    allow_patterns=["p5_ckpt/*_td.pt", "p5_prob/**", "p4_prob/**", "p4_ckpt/*_s*.pt", "src/**", "stats/**",
                    "submissions_p4/**", "submissions_p5/**"])
print("backup done ✅ -> run P5-TDZ")

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


11:39:42 DONE m1_full_first1024_td | 136.6 min
12:35:20 DONE m1_full1024_hier_td | 55.6 min
12:36:37 DONE m1_full1024_hier_s1_td | 1.3 min
12:37:54 DONE m1_full1024_hier_s2_td | 1.3 min
12:37:55 PLAN DONE
09:26:28 DONE m1_full512_hier_s1: dev macro-F1 0.7906 @0.5 | 0.7945 @t=0.32 | attention by decile [0.0581, 0.0238, 0.0462, 0.049, 0.0424, 0.0861, 0.1368, 0.1367, 0.2767, 0.1442] | 3.4 min
09:29:43 DONE m1_full512_hier_s2: dev macro-F1 0.7927 @0.5 | 0.7927 @t=0.50 | attention by decile [0.0659, 0.0255, 0.0498, 0.048, 0.039, 0.0812, 0.1231, 0.1357, 0.2822, 0.1495] | 3.3 min
09:32:16 DONE m1_full1024_hier_s1: dev macro-F1 0.8025 @0.5 | 0.8037 @t=0.47 | attention by decile [0.0351, 0.0069, 0.0258, 0.0901, 0.0163, 0.0704, 0.2537, 0.1912, 0.2585, 0.0519] | 2.5 min
09:34:53 DONE m1_full1024_hier_s2: dev macro-F1 0.7919 @0.5 | 0.7958 @t=0.46 | attention by decile [0.0416, 0.0079, 0.0317, 0.1049, 0.0176, 0.084, 0.2461, 0.1801, 0.2352, 0.0509] | 2.6 min
09:38:14 DONE m1_partial512_hier_s1: dev 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


backup done ✅ -> run P5-TDZ


In [19]:
# ---- P5-TDZ: zips of the train+dev ensembles (thresholds from the dev-validated twins) ----
import sys, json, os, pickle, zipfile, glob, numpy as np
WORK = "/kaggle/working"; sys.path.insert(0, f"{WORK}/src")
import p5_lsi as ls
SYS = json.load(open(f"{WORK}/stats/p5_systems.json"))
test_ids = [m["id"] for m in json.load(open(f"{WORK}/data_p5/clean/test.json"))]
has = lambda m: os.path.exists(f"{WORK}/p5_prob/{m}.pkl")
load = lambda m: pickle.load(open(f"{WORK}/p5_prob/{m}.pkl", "rb"))["test"].astype(np.float32)

def write(name, Pt, T):
    D = ls.decide(Pt, T)
    pred = [{"id": i, "pred": [int(j) for j in np.flatnonzero(d)]} for i, d in zip(test_ids, D)]
    assert len(pred) == len({p["id"] for p in pred}) == len(test_ids) and all(p["pred"] for p in pred)
    folder = f"{WORK}/submissions_p5/p5_{name}"; os.makedirs(folder, exist_ok=True)
    json.dump(pred, open(f"{folder}/prediction.json", "w"))
    with zipfile.ZipFile(f"{folder}/p5_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{folder}/prediction.json", arcname="prediction.json")
    return D.sum(1).mean()

rows = []
for s in ["ens_hier_shrunk", "ens_hier_global", "m1_full1024_hier_s2", "m1_full_first1024"]:
    if s not in SYS:
        continue
    mem = SYS[s]["members"]; mem_td = [m + "_td" if has(m + "_td") else m for m in mem]
    n_td = sum(m.endswith("_td") for m in mem_td)
    if not n_td:
        print(f"{s}: no train+dev members finished -> skipped"); continue
    T = np.array(SYS[s]["T"]); P_old, P_new = np.mean([load(m) for m in mem], 0), np.mean([load(m) for m in mem_td], 0)
    ref = ls.decide(P_old, T).sum(1).mean()                           # labels/case of the dev-validated twin
    lpc = write(f"{s}_td", P_new, T)
    # rate-matched hedge: shift all thresholds so the train+dev system predicts as many labels per case as
    # its validated twin (uses test PREDICTIONS only, never labels) -> protects against a calibration shift
    deltas = np.linspace(-0.2, 0.2, 81)
    d = deltas[int(np.argmin([abs(ls.decide(P_new, np.clip(T + x, 0.02, 0.98)).sum(1).mean() - ref) for x in deltas]))]
    lpc_rm = write(f"{s}_td_rm", P_new, np.clip(T + d, 0.02, 0.98))
    rows.append((s, n_td, len(mem), SYS[s]["dev_honest"], ref, lpc, d, lpc_rm))
print(f"{'system':24s}{'td members':>11s}{'dev (twin)':>11s}{'lab/case twin':>14s}{'td':>7s}{'shift':>7s}{'td_rm':>7s}")
for s, n, k, dv, ref, l1, d, l2 in rows:
    print(f"{s:24s}{f'{n}/{k}':>11s}{dv:>11.4f}{ref:>14.2f}{l1:>7.2f}{d:>+7.3f}{l2:>7.2f}")

# refresh the bundle for the phone uploads: every Phase-5 zip + the new recommended order
first = ["ens_hier_shrunk", "ens_hier_shrunk_td", "ens_hier_shrunk_td_rm"]
lc = json.load(open(f"{WORK}/stats/p5_lc.json")) if os.path.exists(f"{WORK}/stats/p5_lc.json") else {}
if lc.get("name"):
    first.insert(1, lc["name"])
first += ["ens_hier_global_td", "m1_full1024_hier_s2_td", "ens_hier_global"]
old = [l.split("p5_", 1)[1].split(".zip")[0] for l in open(f"{WORK}/submissions_p5/UPLOAD_ORDER.txt")]
allz = {os.path.basename(f)[3:-4]: f for f in glob.glob(f"{WORK}/submissions_p5/p5_*/p5_*.zip")}
order = [n for n in first if n in allz] + [n for n in old if n in allz and n not in first] + \
        sorted(n for n in allz if n not in first and n not in old)
open(f"{WORK}/submissions_p5/UPLOAD_ORDER_v2.txt", "w").write("\n".join(f"{i:2d}. p5_{n}.zip" for i, n in enumerate(order, 1)) + "\n")
with zipfile.ZipFile(f"{WORK}/submissions_p5/ALL_p5_zips_v2.zip", "w") as z:
    for n in order:
        z.write(allz[n], arcname=f"p5_{n}.zip")
    z.write(f"{WORK}/submissions_p5/UPLOAD_ORDER_v2.txt", arcname="UPLOAD_ORDER_v2.txt")
print(f"\nALL_p5_zips_v2.zip: {len(order)} zips | first 8:", order[:8])

from kaggle_secrets import UserSecretsClient                           # back up the new zips + bundle
from huggingface_hub import HfApi
HfApi(token=UserSecretsClient().get_secret("HF_WRITE_TOKEN")).upload_folder(
    repo_id="Utkarsh-1718-1/cs781-phase1-work", repo_type="model", folder_path=WORK,
    commit_message="phase 5 train+dev zips", allow_patterns=["submissions_p5/**", "submissions_p4/**", "stats/**"])
print("backed up ✅ -> download submissions_p5/ALL_p5_zips_v2.zip and the new submissions_p4 zips to Drive")

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


system                   td members dev (twin) lab/case twin     td  shift  td_rm
ens_hier_shrunk                 6/7     0.4300          4.36   4.45 +0.005   4.39
ens_hier_global                 6/7     0.4273          4.10   4.18 +0.005   4.12
m1_full1024_hier_s2             1/1     0.4102          4.09   4.26 +0.015   4.11
m1_full_first1024               1/1     0.3958          4.36   4.26 -0.010   4.38

ALL_p5_zips_v2.zip: 46 zips | first 8: ['ens_hier_shrunk', 'ens_hier_lc_shrunk', 'ens_hier_shrunk_td', 'ens_hier_shrunk_td_rm', 'ens_hier_global_td', 'm1_full1024_hier_s2_td', 'ens_hier_global', 'ens_hier_perlabel']


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

backed up ✅ -> download submissions_p5/ALL_p5_zips_v2.zip and the new submissions_p4 zips to Drive
